# Eight-regime PDE benchmark and diffusion–sorption data contract

This notebook is a cleaned record of the original training program. Execution outputs and notebook execution metadata were removed; computational cells were preserved to maintain the historical experiment protocol. Dataset access and optional cache paths may require configuration.


In [ ]:

# ============================================================
# 0. Runtime / A2 global protocol
# ============================================================
import os, sys, math, time, json, hashlib, shutil, subprocess, zipfile, copy, gc, re
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import h5py
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "A2 broad benchmark requires CUDA."

ROOT=Path(os.getenv("CIDM_BENCH_ROOT", str(Path.cwd())))
OUT=ROOT/"CIDM_Foundation_A2_BroadDynamics_RESULTS"
SRC=OUT/"sources"
FROZEN=OUT/"frozen"
CKPT=OUT/"checkpoints"
FIG=OUT/"figures"
FIGDATA=OUT/"figure_data"
PRED=OUT/"prediction_examples"

# Cache path is fixed before HF restore so compact caches can be restored
# before raw-data download/processing.
CACHE_ROOT=Path(os.getenv(
    "CIDM_A2_CACHE_ROOT",
    "/content/CIDM_A2_BroadDynamics_Cache"
))
DATA_ROOT=Path(os.getenv(
    "CIDM_PDEBENCH_DATA_ROOT",
    "/content/CIDM_PDEBench_Data"
))

for p in [OUT,SRC,FROZEN,CKPT,FIG,FIGDATA,PRED,CACHE_ROOT,DATA_ROOT]:
    p.mkdir(parents=True,exist_ok=True)

HISTORY=4
HORIZON=6
TARGET_PARAMS=500_000
PARAM_TOL=0.05

SEEDS=[17,41,73]
DEV_SEED=17

MAX_UPDATES=int(os.getenv("CIDM_A2_MAX_UPDATES","2500"))
MIN_UPDATES=int(os.getenv("CIDM_A2_MIN_UPDATES","1000"))
EVAL_EVERY=int(os.getenv("CIDM_A2_EVAL_EVERY","100"))
PATIENCE_EVALS=int(os.getenv("CIDM_A2_PATIENCE_EVALS","6"))
REL_MIN_DELTA=float(os.getenv("CIDM_A2_REL_MIN_DELTA","0.001"))

N_TRAIN_TRAJ=int(os.getenv("CIDM_A2_TRAIN_TRAJ","128"))
N_VAL_TRAJ=int(os.getenv("CIDM_A2_VAL_TRAJ","32"))
N_TEST_TRAJ=int(os.getenv("CIDM_A2_TEST_TRAJ","32"))
WINDOWS_PER_TRAJ=int(os.getenv("CIDM_A2_WINDOWS_PER_TRAJ","2"))

BATCH_1D=int(os.getenv("CIDM_A2_BATCH_1D","16"))
BATCH_2D=int(os.getenv("CIDM_A2_BATCH_2D","4"))
WEIGHT_DECAY=1e-4

MODELS=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "Transolver",
]

print("GPU:",torch.cuda.get_device_name(0))
print("Torch:",torch.__version__)
print("A2 output:",OUT)
print("Seeds:",SEEDS)
print("Convergence protocol:",{
    "MAX_UPDATES":MAX_UPDATES,
    "MIN_UPDATES":MIN_UPDATES,
    "EVAL_EVERY":EVAL_EVERY,
    "PATIENCE_EVALS":PATIENCE_EVALS,
    "REL_MIN_DELTA":REL_MIN_DELTA,
})

REPORT={
    "experiment":"CIDM Foundation A2 Broad Dynamics Benchmark",
    "started_at":time.strftime("%Y-%m-%d %H:%M:%S"),
    "protocol":{
        "history":HISTORY,
        "horizon":HORIZON,
        "target_params":TARGET_PARAMS,
        "parameter_tolerance":PARAM_TOL,
        "seeds":SEEDS,
        "max_updates":MAX_UPDATES,
        "min_updates":MIN_UPDATES,
        "eval_every":EVAL_EVERY,
        "patience_evals":PATIENCE_EVALS,
        "relative_min_delta":REL_MIN_DELTA,
        "train_trajectories":N_TRAIN_TRAJ,
        "val_trajectories":N_VAL_TRAJ,
        "test_trajectories":N_TEST_TRAJ,
        "windows_per_trajectory":WINDOWS_PER_TRAJ,
        "precision":"FP32",
        "optimizer":"AdamW",
        "weight_decay":WEIGHT_DECAY,
    }
}

def run(cmd,cwd=None,timeout=1800):
    p=subprocess.run(
        cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,timeout=timeout
    )
    return p.returncode,p.stdout

def md5_file(path,chunk=8*1024*1024):
    h=hashlib.md5()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def sha256_file(path,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def save_json(obj,path):
    Path(path).write_text(
        json.dumps(obj,ensure_ascii=False,indent=2,default=str),
        encoding="utf-8"
    )

def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def portable_state_dict(model):
    raw=model.state_dict()
    out=OrderedDict()
    for k,v in raw.items():
        if k.split(".")[-1]=="_metadata":
            continue
        if torch.is_tensor(v):
            out[k]=v.detach().cpu().clone()
        else:
            out[k]=copy.deepcopy(v)
    if hasattr(raw,"_metadata"):
        out._metadata=copy.deepcopy(raw._metadata)
    return out

def load_portable_state_dict(model,sd):
    incompat=model.load_state_dict(sd,strict=False)
    missing=list(incompat.missing_keys)
    unexpected=[
        k for k in incompat.unexpected_keys
        if k.split(".")[-1]!="_metadata"
    ]
    if missing or unexpected:
        raise RuntimeError(
            f"checkpoint mismatch: missing={missing}, unexpected={unexpected}"
        )


In [ ]:

# ============================================================
# 1. Freeze audit:
#    CIDM Foundation Core v1.0 Final
#    + CIDM-Complete archived executable topology
# ============================================================
CIDM_CORE_SOURCE = '"""\nCIDM Foundation Core v1.0\n=========================\n\nFrozen canonical implementation for fair external benchmarking.\n\nThis file contains ONLY the universal core selected after the A1-R1-R3\ncomponent-attribution review:\n\n1. Finite-history predictive information state.\n2. Spatial endpoint encoder.\n3. Temporal-history encoder.\n4. Fusion into a single latent information state.\n5. Bounded local latent residual dynamics.\n6. Bounded positive step size.\n7. Pure latent rollout.\n8. Field decoder.\n\nNot part of the universal core:\n- phase/spectral encoder,\n- skew-symmetric channel rotation,\n- antisymmetric flux,\n- hidden reservoir,\n- causal forcing port,\n- transport branch,\n- cross-scale transfer,\n- BER/SOC,\n- state-sync regularization,\n- Cayley update.\n\nThose are task-specific or training/numerical optionals and must not be silently\nenabled in Foundation Core benchmark runs.\n\nImportant:\nThe numerical caps below are architectural hyperparameters, not physical\nconstants.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass(frozen=True)\nclass CIDMFoundationConfig:\n    spatial_dim: int = 2          # 1 or 2\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n\n    # Frozen benchmark defaults; architectural hyperparameters, not laws.\n    local_cap: float = 0.10\n    step_cap: float = 0.30\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef _conv_nd(dim: int):\n    if dim == 1:\n        return nn.Conv1d\n    if dim == 2:\n        return nn.Conv2d\n    raise ValueError(f"spatial_dim must be 1 or 2, got {dim}")\n\n\nclass ConvBlockND(nn.Module):\n    def __init__(self, dim: int, in_ch: int, out_ch: int, hidden: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, out_ch, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field X_t."""\n\n    def __init__(self, dim: int, channels: int, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, channels, latent_dim, hidden)\n\n    def forward(self, x_t: torch.Tensor) -> torch.Tensor:\n        return self.net(x_t)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode finite history H_t = X_{t-L+1:t}."""\n\n    def __init__(\n        self,\n        dim: int,\n        channels: int,\n        hist_len: int,\n        latent_dim: int,\n        hidden: int,\n    ):\n        super().__init__()\n        self.hist_len = hist_len\n        self.channels = channels\n        self.net = ConvBlockND(\n            dim,\n            channels * hist_len,\n            latent_dim,\n            hidden,\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: [B, L, C, *S]\n        if history.shape[1] != self.hist_len:\n            raise ValueError(\n                f"Expected history length {self.hist_len}, got {history.shape[1]}"\n            )\n        b, l, c, *spatial = history.shape\n        if c != self.channels:\n            raise ValueError(f"Expected {self.channels} channels, got {c}")\n        x = history.reshape(b, l * c, *spatial)\n        return self.net(x)\n\n\nclass InformationStateFusion(nn.Module):\n    """Fuse spatial endpoint and temporal-history views into Psi_t."""\n\n    def __init__(self, dim: int, latent_dim: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(2 * latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            Conv(latent_dim, latent_dim, 1),\n        )\n\n    def forward(\n        self,\n        z_space: torch.Tensor,\n        z_time: torch.Tensor,\n    ) -> torch.Tensor:\n        return self.net(torch.cat([z_space, z_time], dim=1))\n\n\nclass BoundedLocalResidualDynamics(nn.Module):\n    """\n    Minimal universal transition field.\n\n    The residual branch is bounded twice:\n    - branch responsibility: local_cap * sigmoid(local_logit)\n    - global step: step_cap * sigmoid(step_logit)\n\n    This prevents an unconstrained transition from silently dominating the\n    latent information state.\n    """\n\n    def __init__(\n        self,\n        dim: int,\n        latent_dim: int,\n        hidden: int,\n        local_cap: float,\n        step_cap: float,\n    ):\n        super().__init__()\n        self.local_cap = float(local_cap)\n        self.step_cap = float(step_cap)\n\n        self.local_net = ConvBlockND(\n            dim, latent_dim, latent_dim, hidden\n        )\n        self.local_logit = nn.Parameter(torch.tensor(-2.2))\n        self.step_logit = nn.Parameter(torch.tensor(-1.5))\n\n    def vector_field(self, psi: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        local_scale = self.local_cap * torch.sigmoid(self.local_logit)\n        raw = torch.tanh(self.local_net(psi))\n        delta_local = local_scale * raw\n\n        info = {\n            "local_scale": local_scale,\n            "local_strength": delta_local.abs().mean(),\n        }\n        return delta_local, info\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        delta_local, info = self.vector_field(psi)\n        step = self.step_cap * torch.sigmoid(self.step_logit)\n        psi_next = psi + step * delta_local\n\n        info = {\n            **info,\n            "step_size": step,\n            "delta_strength": (step * delta_local).abs().mean(),\n        }\n        return psi_next, info\n\n\nclass FieldDecoder(nn.Module):\n    def __init__(self, dim: int, latent_dim: int, channels: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, latent_dim, channels, hidden)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass CIDMFoundationCore(nn.Module):\n    """\n    Closed Information Dynamics Model — Foundation Core v1.0.\n\n    Input:\n        history: [B, L, C, *S]\n\n    Output dict:\n        pred:       [B, K, C, *S]\n        latent0:    [B, D, *S]\n        latent:     [B, D, *S]\n        latent_seq: [B, K, D, *S]\n        diagnostics\n    """\n\n    VERSION = "CIDM-Foundation-Core-v1.0"\n\n    def __init__(self, config: CIDMFoundationConfig):\n        super().__init__()\n        self.config = config\n\n        d = config.spatial_dim\n        D = config.latent_dim\n        H = config.hidden\n\n        self.space_encoder = SpaceEncoder(\n            d, config.channels, D, H\n        )\n        self.time_encoder = TimeHistoryEncoder(\n            d, config.channels, config.hist_len, D, H\n        )\n        self.fusion = InformationStateFusion(d, D)\n\n        self.dynamics = BoundedLocalResidualDynamics(\n            d, D, H,\n            local_cap=config.local_cap,\n            step_cap=config.step_cap,\n        )\n        self.decoder = FieldDecoder(\n            d, D, config.channels, H\n        )\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        z_space = self.space_encoder(last)\n        z_time = self.time_encoder(history)\n        return self.fusion(z_space, z_time)\n\n    def step(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        return self.dynamics(psi)\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi)\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        K = int(horizon or self.config.horizon)\n\n        psi0 = self.encode(history)\n        psi = psi0\n\n        preds = []\n        states = []\n        local_strengths = []\n        step_sizes = []\n        delta_strengths = []\n\n        spatial_dims = tuple(range(1, psi.ndim))\n        energy0 = (psi ** 2).mean(dim=spatial_dims)\n        energies = []\n\n        for _ in range(K):\n            psi, info = self.step(psi)\n            y = self.decode(psi)\n\n            preds.append(y)\n            states.append(psi)\n            local_strengths.append(info["local_strength"])\n            step_sizes.append(info["step_size"])\n            delta_strengths.append(info["delta_strength"])\n            energies.append((psi ** 2).mean(dim=spatial_dims))\n\n        pred = torch.stack(preds, dim=1)\n        latent_seq = torch.stack(states, dim=1)\n        energy = torch.stack(energies, dim=1)\n\n        diagnostics: Dict[str, torch.Tensor] = {\n            "local_strength": torch.stack(local_strengths).mean(),\n            "step_size": torch.stack(step_sizes).mean(),\n            "delta_strength": torch.stack(delta_strengths).mean(),\n            "latent_energy_drift": (energy[:, -1] - energy0).abs().mean(),\n        }\n\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n\n        return {\n            "pred": pred,\n            "latent0": psi0,\n            "latent": psi,\n            "latent_seq": latent_seq,\n            "diagnostics": diagnostics,\n        }\n\n\ndef build_cidm_foundation_core(\n    *,\n    spatial_dim: int,\n    channels: int,\n    hist_len: int,\n    horizon: int,\n    latent_dim: int,\n    hidden: int,\n    local_cap: float = 0.10,\n    step_cap: float = 0.30,\n) -> CIDMFoundationCore:\n    cfg = CIDMFoundationConfig(\n        spatial_dim=spatial_dim,\n        channels=channels,\n        hist_len=hist_len,\n        horizon=horizon,\n        latent_dim=latent_dim,\n        hidden=hidden,\n        local_cap=local_cap,\n        step_cap=step_cap,\n    )\n    return CIDMFoundationCore(cfg)\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n\n    # 1D smoke\n    m1 = build_cidm_foundation_core(\n        spatial_dim=1,\n        channels=1,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x1 = torch.randn(2, 4, 1, 64)\n    y1 = m1(x1)\n    assert y1["pred"].shape == (2, 6, 1, 64)\n\n    # 2D smoke\n    m2 = build_cidm_foundation_core(\n        spatial_dim=2,\n        channels=2,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x2 = torch.randn(2, 4, 2, 24, 24)\n    y2 = m2(x2)\n    assert y2["pred"].shape == (2, 6, 2, 24, 24)\n\n    print("CIDM Foundation Core v1.0 smoke PASS")\n    print("1D params:", count_parameters(m1))\n    print("2D params:", count_parameters(m2))\n'
CIDM_FREEZE_MANIFEST = '{\n  "name": "CIDM Foundation Core",\n  "version": "v1.0 Final",\n  "freeze_basis": "A1-R1-R3 Core Component Causal Attribution & Freeze",\n  "freeze_gate": "PASS_TO_CORE_FINAL_FREEZE",\n  "default_core": {\n    "encoders": [\n      "SpaceEncoder",\n      "TimeHistoryEncoder"\n    ],\n    "fusion": "InformationStateFusion",\n    "dynamics": "BoundedLocalResidualDynamics",\n    "step": "bounded positive residual step",\n    "decoder": "FieldDecoder",\n    "rollout": "pure latent rollout",\n    "default_training": "prediction / multi-step rollout loss"\n  },\n  "evidence_summary": {\n    "NoSpaceInfo_mean_rmse_effect_pct": 4.895219760746574,\n    "NoTimeInfo_mean_rmse_effect_pct": 5.171493884445902,\n    "NoPhaseInfo_mean_rmse_effect_pct": -2.79857874424354,\n    "NoSkew_mean_rmse_effect_pct": -0.521169227349116,\n    "NoFlux_mean_rmse_effect_pct": -0.9437659673091198,\n    "NoLocal_mean_rmse_effect_pct": 2.397827861124943,\n    "BlackboxCore_mean_rmse_effect_pct": 1.7607338250780888,\n    "DirectPredictor_mean_rmse_effect_pct": 14.230675545900779,\n    "FreeChannelMix_vs_Skew_rmse_change_pct": 3.476865940431788,\n    "FreeFlux_vs_AntisymmetricFlux_rmse_change_pct": -2.5548147465622244,\n    "Sync05_vs_PredOnly_rmse_change_pct": 10.445702293374174,\n    "Sync05_sync_gain_change": 0.5611278432875664\n  },\n  "important_cautions": [\n    "DirectPredictor in A1-R1-R3 was materially smaller than Full, so its 14.23% RMSE gap is supportive rather than a parameter-matched causal estimate.",\n    "Only two seeds were used; several component effects had high seed variance.",\n    "Space/Time encoders show cross-task value but Diffusion-Reaction did not consistently require them.",\n    "The numerical cap values are architectural hyperparameters, not physical constants."\n  ],\n  "optional_components": {\n    "PhaseSpectralEncoder": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "NoPhaseInfo improved mean RMSE by 2.80%; benefits were inconsistent across tasks/seeds."\n    },\n    "SkewSymmetricChannelRotation": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "Removing skew slightly improved mean RMSE; however skew-constrained mixing was better than free channel mixing when a mixing branch was present."\n    },\n    "AntisymmetricFlux": {\n      "status": "SPECIALIZED CONSERVATIVE-EXCHANGE OPTIONAL",\n      "reason": "NoFlux slightly improved mean RMSE and FreeFlux outperformed antisymmetric flux on average."\n    },\n    "HiddenReservoir": {\n      "status": "OPTIONAL",\n      "reason": "Use for unresolved memory/internal closure; not part of the R3 universal core."\n    },\n    "NativeCausalPort": {\n      "status": "OPTIONAL",\n      "reason": "Use only when known causal forcing is available."\n    },\n    "Transport": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for displacement/coherent-structure dominated systems."\n    },\n    "CrossScaleTransfer": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for tasks with explicit cross-scale transfer."\n    },\n    "StateSyncRegularizer": {\n      "status": "TRAINING OPTIONAL",\n      "reason": "Improved sync metric but increased mean RMSE by 10.45% in R3."\n    },\n    "CayleySkewUpdate": {\n      "status": "NUMERICAL OPTIONAL",\n      "reason": "Strictly orthogonal discretization if a skew branch is deliberately enabled."\n    },\n    "BER_SOC": {\n      "status": "SPECIALIZED EXTENSION FAMILY",\n      "reason": "Keep outside Foundation Core; appropriate for open/physical closure studies."\n    }\n  }\n}'
CIDM_COMPLETE_ARCHIVE_SOURCE = '"""\nCIDM architecture archive\n=========================\n\nReference PyTorch implementation for the finalized model family:\n\n1. CIDM / Closed Information Dynamics Model\n   - General-purpose closed information dynamics backbone.\n   - Corresponds conceptually to the previous experimental `sticm_v3_full`.\n\n2. CIDM-BER / CIDM with Boundary Energy Reservoir\n   - Physics-closure enhanced variant.\n   - Adds a latent Spectral Operator Core and a Boundary Energy Reservoir (BER)\n     residual closure layer.\n   - Corresponds conceptually to the previous experimental `sticm_soc_ber`.\n\nThis file is intended for architecture archiving and future experiment reuse. It\ncontains only model definitions and lightweight diagnostics; training/evaluation\nloops are intentionally kept outside this file.\n\nExpected input format\n---------------------\n`history`: Tensor[B, L, C, H, W]\n`port`: optional Tensor[B, C_port, H, W] or Tensor[B, horizon, C_port, H, W]\n`target`: optional Tensor[B, horizon, C, H, W] for diagnostics/loss construction\n\nOutput dictionary\n-----------------\n`pred`: Tensor[B, horizon, C, H, W]\n`latent`: final latent state\n`latent_seq`: Tensor[B, horizon, D, H, W]\n`diagnostics`: scalar tensors for energy drift, SOC strength, BER strength, etc.\n\nAuthoring note\n--------------\nThis is a cleaned reference architecture. Exact Colab benchmark implementations\nmay contain additional data-generation, plotting, logging, and ablation utilities.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass CIDMConfig:\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n    port_channels: int = 1\n    use_phase_encoder: bool = True\n    use_time_encoder: bool = True\n    use_hidden_reservoir: bool = True\n    use_flux: bool = True\n    use_reverse_diagnostics: bool = True\n    # CIDM-BER options\n    soc_modes1: int = 8\n    soc_modes2: int = 8\n    ber_add_scale: float = 0.05\n    ber_enabled: bool = True\n    soc_enabled: bool = True\n\n\ndef count_parameters(model: nn.Module) -> int:\n    """Return the number of trainable parameters."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef conv_block(in_ch: int, out_ch: int, hidden: int, groups: int = 1) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv2d(in_ch, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, out_ch, 1),\n    )\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field into latent information state."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode a finite observation history as temporal context."""\n\n    def __init__(self, channels: int, hist_len: int, latent_dim: int):\n        super().__init__()\n        self.hist_len = hist_len\n        self.net = nn.Sequential(\n            nn.Conv2d(channels * hist_len, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: B, L, C, H, W\n        b, l, c, h, w = history.shape\n        if l != self.hist_len:\n            raise ValueError(f"Expected hist_len={self.hist_len}, got {l}.")\n        return self.net(history.reshape(b, l * c, h, w))\n\n\nclass PhaseSpectralEncoder(nn.Module):\n    """FFT-based phase/amplitude encoder projected back to the spatial latent grid."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.freq_net = nn.Sequential(\n            nn.Conv2d(channels * 3, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.spatial_net = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(x, dim=(-2, -1), norm="ortho")\n        amp = torch.log1p(torch.abs(z))\n        feats = torch.cat([z.real, z.imag, amp], dim=1)\n        f = self.freq_net(feats)\n        # Return to the spatial support while retaining frequency-aware features.\n        f_complex = torch.complex(f, torch.zeros_like(f))\n        spatial = torch.fft.ifft2(f_complex, dim=(-2, -1), norm="ortho").real\n        return self.spatial_net(spatial)\n\n\nclass FieldDecoder(nn.Module):\n    """Decode latent information state to physical/observed field."""\n\n    def __init__(self, latent_dim: int, channels: int, hidden: int = 64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, channels, 1),\n        )\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass SpectralFieldDecoder(nn.Module):\n    """Small spectral residual decoder used by CIDM for phase-aware readout."""\n\n    def __init__(self, latent_dim: int, channels: int):\n        super().__init__()\n        self.mix = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, channels, 1),\n        )\n        self.scale = nn.Parameter(torch.tensor(-2.2))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(psi, dim=(-2, -1), norm="ortho")\n        # Simple low-pass emphasis for phase-stable readout.\n        H, W = psi.shape[-2:]\n        ky = torch.fft.fftfreq(H, device=psi.device).view(1, 1, H, 1)\n        kx = torch.fft.fftfreq(W, device=psi.device).view(1, 1, 1, W)\n        filt = torch.exp(-6.0 * (kx ** 2 + ky ** 2))\n        low = torch.fft.ifft2(z * filt, dim=(-2, -1), norm="ortho").real\n        return torch.sigmoid(self.scale) * self.mix(low)\n\n\nclass AntisymmetricFlux(nn.Module):\n    """Conservative local exchange field.\n\n    For each neighbor pair, the edge update is constructed as f(a,b)-f(b,a),\n    then added to one side and subtracted from the other. This encodes the idea\n    that local exchanges can redistribute information without creating it.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.edge_net = nn.Sequential(\n            nn.Conv2d(2 * latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n\n    def edge(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n        return self.edge_net(torch.cat([a, b], dim=1)) - self.edge_net(torch.cat([b, a], dim=1))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        out = torch.zeros_like(psi)\n        # x direction\n        right = torch.roll(psi, shifts=-1, dims=-1)\n        fx = self.edge(psi, right)\n        out = out + fx - torch.roll(fx, shifts=1, dims=-1)\n        # y direction; safe for H=1 because roll is identity and antisymmetry cancels.\n        down = torch.roll(psi, shifts=-1, dims=-2)\n        fy = self.edge(psi, down)\n        out = out + fy - torch.roll(fy, shifts=1, dims=-2)\n        return out\n\n\nclass SkewChannelRotation(nn.Module):\n    """Learned skew-symmetric channel mixing for approximately conservative flow."""\n\n    def __init__(self, latent_dim: int):\n        super().__init__()\n        self.raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        A = self.raw - self.raw.t()\n        return torch.einsum("ij,bjhw->bihw", A, psi)\n\n\nclass HiddenReservoirTransfer(nn.Module):\n    """Internal hidden reservoir transfer for unobserved latent exchange."""\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = conv_block(latent_dim, latent_dim, hidden)\n        self.scale = nn.Parameter(torch.tensor(-2.0))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.scale) * self.net(psi)\n\n\nclass ClosedInformationDynamicsCore(nn.Module):\n    """CIDM closed latent vector field."""\n\n    def __init__(self, latent_dim: int, hidden: int, use_flux: bool = True, use_hidden_reservoir: bool = True):\n        super().__init__()\n        self.use_flux = use_flux\n        self.use_hidden_reservoir = use_hidden_reservoir\n        self.skew = SkewChannelRotation(latent_dim)\n        self.flux = AntisymmetricFlux(latent_dim, hidden)\n        self.hidden = HiddenReservoirTransfer(latent_dim, hidden)\n        self.local = conv_block(latent_dim, latent_dim, hidden)\n        self.step_scale = nn.Parameter(torch.tensor(0.18))\n\n    def vector_field(self, psi: torch.Tensor) -> torch.Tensor:\n        dpsi = self.skew(psi)\n        if self.use_flux:\n            dpsi = dpsi + self.flux(psi)\n        if self.use_hidden_reservoir:\n            dpsi = dpsi + 0.25 * self.hidden(psi)\n        # A small local residual stabilizes non-Hamiltonian observed fields.\n        dpsi = dpsi + 0.10 * self.local(psi)\n        return dpsi\n\n    def forward(self, psi: torch.Tensor, dt_scale: float = 1.0) -> torch.Tensor:\n        alpha = torch.tanh(self.step_scale) * dt_scale\n        return psi + alpha * self.vector_field(psi)\n\n\nclass LatentSpectralOperatorCore(nn.Module):\n    """Latent-domain spectral operator for CIDM-BER.\n\n    This is a residual spectral vector field inside the latent dynamics, not an\n    external FNO baseline. It learns complex low-frequency mode mixing and maps\n    it back to the latent grid as a global correction to local closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int = 64, modes1: int = 8, modes2: int = 8):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.modes1 = modes1\n        self.modes2 = modes2\n        scale = 1.0 / max(1, latent_dim)\n        self.weight_pos = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.weight_neg = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.pre = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.post = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-1.8))\n\n    @staticmethod\n    def compl_mul2d(x: torch.Tensor, wri: torch.Tensor) -> torch.Tensor:\n        w = torch.view_as_complex(wri.contiguous())\n        return torch.einsum("bihw,iohw->bohw", x, w)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        B, D, H, W = psi.shape\n        x = self.pre(psi)\n        x_ft = torch.fft.rfft2(x, dim=(-2, -1), norm="ortho")\n        out_ft = torch.zeros(B, D, H, W // 2 + 1, dtype=torch.cfloat, device=psi.device)\n        m1 = min(self.modes1, H)\n        m2 = min(self.modes2, W // 2 + 1)\n        out_ft[:, :, :m1, :m2] = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weight_pos[:, :, :m1, :m2])\n        if H > 1:\n            out_ft[:, :, -m1:, :m2] = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weight_neg[:, :, :m1, :m2])\n        y = torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1), norm="ortho")\n        return torch.sigmoid(self.logit_scale) * self.post(y)\n\n\nclass BoundaryEnergyReservoir(nn.Module):\n    """Boundary Energy Reservoir (BER) residual closure layer.\n\n    BER is a virtual external energy reservoir, not a full external-state model.\n    It learns a minimal residual exchange field that accounts for open-system\n    effects not captured by the internal closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, port_channels: int, hidden: int):\n        super().__init__()\n        self.port_channels = max(1, port_channels)\n        in_ch = latent_dim * 2 + self.port_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 3, padding=1),\n        )\n        mask_hidden = max(8, hidden // 2)\n        self.mask = nn.Sequential(\n            nn.Conv2d(in_ch, mask_hidden, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(mask_hidden, 1, 1),\n            nn.Sigmoid(),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-3.0))\n\n    def _port_to_grid(self, psi: torch.Tensor, port: Optional[torch.Tensor]) -> torch.Tensor:\n        B, _, H, W = psi.shape\n        if port is None or port.numel() == 0:\n            return torch.zeros(B, self.port_channels, H, W, device=psi.device, dtype=psi.dtype)\n        if port.dim() != 4:\n            raise ValueError("port must be Tensor[B, C_port, H, W] for one step.")\n        p = port\n        if p.shape[-2:] != (H, W):\n            p = F.interpolate(p, size=(H, W), mode="bilinear", align_corners=False)\n        if p.shape[1] < self.port_channels:\n            pad = torch.zeros(B, self.port_channels - p.shape[1], H, W, device=psi.device, dtype=psi.dtype)\n            p = torch.cat([p, pad], dim=1)\n        elif p.shape[1] > self.port_channels:\n            p = p[:, : self.port_channels]\n        return p\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n        internal_delta: torch.Tensor,\n        port: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        p = self._port_to_grid(psi, port)\n        x = torch.cat([psi, internal_delta, p], dim=1)\n        raw = torch.tanh(self.net(x))\n        mask = self.mask(x)\n        scale = torch.sigmoid(self.logit_scale)\n        ber_delta = scale * mask * raw\n        info = {\n            "ber_strength": ber_delta.abs().mean(),\n            "ber_mask_mean": mask.mean(),\n            "ber_scale": scale,\n        }\n        return ber_delta, info\n\n\nclass CIDM(nn.Module):\n    """Closed Information Dynamics Model.\n\n    General-purpose主体架构. This version corresponds to the stabilized closed\n    dynamics backbone: spatial information encoding, temporal history encoding,\n    phase-spectral feature encoding, conservative local exchange, internal hidden\n    reservoir transfer, and autoregressive latent rollout.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__()\n        self.config = config\n        D = config.latent_dim\n        H = config.hidden\n        self.space_encoder = SpaceEncoder(config.channels, D)\n        self.time_encoder = TimeHistoryEncoder(config.channels, config.hist_len, D)\n        self.phase_encoder = PhaseSpectralEncoder(config.channels, D)\n        fuse_inputs = D + (D if config.use_time_encoder else 0) + (D if config.use_phase_encoder else 0)\n        self.fuse = nn.Sequential(nn.Conv2d(fuse_inputs, D, 1), nn.SiLU(), nn.Conv2d(D, D, 1))\n        self.core = ClosedInformationDynamicsCore(D, H, use_flux=config.use_flux, use_hidden_reservoir=config.use_hidden_reservoir)\n        self.decoder = FieldDecoder(D, config.channels, H)\n        self.spectral_decoder = SpectralFieldDecoder(D, config.channels)\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        feats = [self.space_encoder(last)]\n        if self.config.use_time_encoder:\n            feats.append(self.time_encoder(history))\n        if self.config.use_phase_encoder:\n            feats.append(self.phase_encoder(last))\n        return self.fuse(torch.cat(feats, dim=1))\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi) + self.spectral_decoder(psi)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_next = self.core(psi)\n        return psi_next, {"soc_strength": torch.zeros((), device=psi.device), "ber_strength": torch.zeros((), device=psi.device)}\n\n    def _select_port_step(self, port: Optional[torch.Tensor], t: int) -> Optional[torch.Tensor]:\n        if port is None:\n            return None\n        if port.dim() == 5:\n            return port[:, min(t, port.shape[1] - 1)]\n        return port\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        port: Optional[torch.Tensor] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        horizon = int(horizon or self.config.horizon)\n        psi0 = self.encode(history)\n        psi = psi0\n        preds = []\n        latent_seq = []\n        soc_strengths, ber_strengths = [], []\n        energy0 = (psi0 ** 2).mean(dim=(1, 2, 3))\n        energies = []\n        for t in range(horizon):\n            psi, info = self.step(psi, t, self._select_port_step(port, t))\n            y = self.decode(psi)\n            preds.append(y)\n            latent_seq.append(psi)\n            energies.append((psi ** 2).mean(dim=(1, 2, 3)))\n            soc_strengths.append(info.get("soc_strength", torch.zeros((), device=psi.device)))\n            ber_strengths.append(info.get("ber_strength", torch.zeros((), device=psi.device)))\n        pred = torch.stack(preds, dim=1)\n        latent = torch.stack(latent_seq, dim=1)\n        energy = torch.stack(energies, dim=1)\n        diagnostics = {\n            "energy_drift": (energy[:, -1] - energy0).abs().mean(),\n            "soc_strength": torch.stack([s if s.dim() else s.expand(1) for s in soc_strengths]).mean(),\n            "ber_strength": torch.stack([b if b.dim() else b.expand(1) for b in ber_strengths]).mean(),\n        }\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n        return {"pred": pred, "latent": psi, "latent_seq": latent, "diagnostics": diagnostics}\n\n\nclass CIDM_BER(CIDM):\n    """CIDM with Spectral Operator Core and Boundary Energy Reservoir.\n\n    Physics-enhanced variant for systems with salient wave/flow propagation,\n    quasi-closed dynamics, boundary exchange, or energy closure requirements.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__(config)\n        self.soc = LatentSpectralOperatorCore(\n            config.latent_dim,\n            hidden=config.hidden,\n            modes1=config.soc_modes1,\n            modes2=config.soc_modes2,\n        )\n        self.ber = BoundaryEnergyReservoir(config.latent_dim, config.port_channels, config.hidden)\n        self.ber_add_scale = float(config.ber_add_scale)\n        self.soc_enabled = bool(config.soc_enabled)\n        self.ber_enabled = bool(config.ber_enabled)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_int = self.core(psi)\n        soc_delta = self.soc(psi) if self.soc_enabled else torch.zeros_like(psi)\n        psi_int = psi_int + soc_delta\n        internal_delta = psi_int - psi\n        if self.ber_enabled:\n            ber_delta, ber_info = self.ber(psi, internal_delta.detach(), port)\n            psi_next = psi_int + self.ber_add_scale * ber_delta\n        else:\n            ber_delta = torch.zeros_like(psi)\n            ber_info = {"ber_strength": torch.zeros((), device=psi.device), "ber_mask_mean": torch.zeros((), device=psi.device)}\n            psi_next = psi_int\n        info = {\n            "soc_strength": soc_delta.abs().mean(),\n            "ber_strength": ber_info["ber_strength"],\n            "ber_mask_mean": ber_info.get("ber_mask_mean", torch.zeros((), device=psi.device)),\n            "ber_budget_proxy": (ber_delta.mean(dim=(1, 2, 3)) - internal_delta.mean(dim=(1, 2, 3))).abs().mean(),\n        }\n        return psi_next, info\n\n\ndef build_cidm(config: Optional[CIDMConfig] = None) -> CIDM:\n    return CIDM(config or CIDMConfig())\n\n\ndef build_cidm_ber(config: Optional[CIDMConfig] = None) -> CIDM_BER:\n    cfg = config or CIDMConfig()\n    return CIDM_BER(cfg)\n\n\n# Backward-compatible aliases for previous internal experiment names.\nClosedInformationDynamicsModel = CIDM\nClosedInformationDynamicsBERModel = CIDM_BER\nSTICM_V3_FULL_ALIAS = CIDM\nSTICM_SOC_BER_ALIAS = CIDM_BER\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    cfg = CIDMConfig(channels=1, hist_len=4, horizon=1, latent_dim=4, hidden=8, port_channels=1)\n    x = torch.randn(1, cfg.hist_len, cfg.channels, 4, 4)\n    port = torch.randn(1, cfg.port_channels, 4, 4)\n    for cls in [CIDM, CIDM_BER]:\n        model = cls(cfg)\n        out = model(x, port=port)\n        print(cls.__name__, out["pred"].shape, "params=", count_parameters(model), out["diagnostics"])\n'

EXPECTED_CORE_SHA256 = "e5e26b35765301e9ecd25aec5a6c507395ff370dca11c820f540ff0f1e79f510"
EXPECTED_MANIFEST_SHA256 = "9508dd000005a154d3d3fa4437602d4214a6266d630bf13af855e5517649ecbd"
EXPECTED_COMPLETE_ARCHIVE_SHA256 = "f9b1cf4ff7630f5540bb3adde20e4c4b81fff3218d9325ea3105bf28b1d97665"

core_file=FROZEN/"cidm_foundation_core_v1.py"
manifest_file=FROZEN/"CIDM_Foundation_Core_v1_0_FREEZE_MANIFEST.json"
complete_file=FROZEN/"cidm_complete_archived_topology.py"

core_file.write_text(CIDM_CORE_SOURCE,encoding="utf-8")
manifest_file.write_text(CIDM_FREEZE_MANIFEST,encoding="utf-8")
complete_file.write_text(CIDM_COMPLETE_ARCHIVE_SOURCE,encoding="utf-8")

core_sha=sha256_file(core_file)
manifest_sha=sha256_file(manifest_file)
complete_sha=sha256_file(complete_file)

print("CIDM Core SHA256:",core_sha)
print("Core Manifest SHA256:",manifest_sha)
print("CIDM-Complete archive SHA256:",complete_sha)

assert core_sha==EXPECTED_CORE_SHA256
assert manifest_sha==EXPECTED_MANIFEST_SHA256
assert complete_sha==EXPECTED_COMPLETE_ARCHIVE_SHA256

import importlib.util

spec=importlib.util.spec_from_file_location("cidm_foundation_core_v1",core_file)
cidm_mod=importlib.util.module_from_spec(spec)
sys.modules["cidm_foundation_core_v1"]=cidm_mod
spec.loader.exec_module(cidm_mod)

CIDMFoundationCore=cidm_mod.CIDMFoundationCore
CIDMFoundationConfig=cidm_mod.CIDMFoundationConfig
build_cidm_foundation_core=cidm_mod.build_cidm_foundation_core

spec2=importlib.util.spec_from_file_location("cidm_complete_archive",complete_file)
cidm_complete_mod=importlib.util.module_from_spec(spec2)
sys.modules["cidm_complete_archive"]=cidm_complete_mod
spec2.loader.exec_module(cidm_complete_mod)

CIDMCompleteConfig=cidm_complete_mod.CIDMConfig
CIDMCompleteArchived=cidm_complete_mod.CIDM_BER

# Core smoke
m=build_cidm_foundation_core(
    spatial_dim=2,channels=1,hist_len=4,horizon=6,
    latent_dim=16,hidden=32
).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 2D smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=2,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 1D-via-H=1 smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=1,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,1,64,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,1,64)
del m,x,y
torch.cuda.empty_cache()

COMPLETE_CONTRACT={
    "benchmark_name":"CIDM-Complete",
    "executable_mapping":"cidm_architectures.py::CIDM_BER",
    "archive_sha256":complete_sha,
    "components":[
        "SpaceEncoder","TimeHistoryEncoder","PhaseSpectralEncoder",
        "SkewChannelRotation","AntisymmetricFlux","HiddenReservoirTransfer",
        "LocalResidual","LatentSpectralOperatorCore","BoundaryEnergyReservoir",
        "FieldDecoder","SpectralFieldDecoder","PureLatentRollout"
    ],
    "known_forcing_port":"disabled / None because PDEBench headline tasks provide no common forcing field",
    "static_fields":"not supplied; absent from archived executable implementation",
    "1d_protocol":"lift to H=1, run exact archived Conv2D implementation, squeeze H=1",
    "future_target_at_inference":"forbidden",
    "note":"Figure-level State-Observation Closure is not invented here; archived executable SOC is LatentSpectralOperatorCore."
}
save_json(COMPLETE_CONTRACT,OUT/"CIDM_Complete_Benchmark_Contract.json")

REPORT["cidm_freeze"]={
    "core_sha256":core_sha,
    "manifest_sha256":manifest_sha,
    "exact":True,
}
REPORT["cidm_complete_freeze"]={
    "archive_sha256":complete_sha,
    "exact":True,
    "contract":COMPLETE_CONTRACT,
}

print("FROZEN CIDM CORE: PASS")
print("FROZEN CIDM-COMPLETE ARCHIVE: PASS")



# 2. Exact official source checkout

固定此前 provenance qualification 已验证过的 exact commit：

```text
PDEBench       4ff3e3a4aa1561721b5571fa3a048a0a463e0568
NeuralOperator 00b7d86f8d74ff0af55da53eb585fe26df9c71f0
Transolver     75e0f67643806a81cd1d3f6adc88dd8c02416fe7
```

正式 Benchmark 不追随 `main/master`，防止实验期间 upstream 漂移。


In [ ]:

# ============================================================
# 2. Exact source checkout + official imports
# ============================================================
LOCKED={
    "pdebench":{
        "url":"https://github.com/pdebench/PDEBench.git",
        "sha":"4ff3e3a4aa1561721b5571fa3a048a0a463e0568",
    },
    "neuraloperator":{
        "url":"https://github.com/neuraloperator/neuraloperator.git",
        "sha":"00b7d86f8d74ff0af55da53eb585fe26df9c71f0",
    },
    "transolver":{
        "url":"https://github.com/thuml/Transolver.git",
        "sha":"75e0f67643806a81cd1d3f6adc88dd8c02416fe7",
    },
}
checkout=[]
for rid,m in LOCKED.items():
    dst=SRC/rid
    if dst.exists():
        rc,head=run(["git","rev-parse","HEAD"],cwd=dst,timeout=60)
        if rc==0 and head.strip()==m["sha"]:
            print("[reuse source]",rid,head.strip())
            checkout.append({"id":rid,"head":head.strip(),"exact":True})
            continue
        shutil.rmtree(dst)

    rc,o=run(["git","clone","--filter=blob:none","--no-checkout",m["url"],str(dst)])
    if rc: raise RuntimeError(f"{rid} clone failed\n{o[-3000:]}")
    rc,o=run(["git","fetch","origin",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} fetch failed\n{o[-3000:]}")
    rc,o=run(["git","checkout","--detach",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} checkout failed\n{o[-3000:]}")
    _,head=run(["git","rev-parse","HEAD"],cwd=dst)
    head=head.strip()
    checkout.append({"id":rid,"head":head,"exact":head==m["sha"]})
    print(rid,head,"PASS" if head==m["sha"] else "FAIL")

assert all(x["exact"] for x in checkout)
pd.DataFrame(checkout).to_csv(OUT/"source_checkout.csv",index=False)

# Dependencies needed by the exact official cores.
rc,o=run([
    sys.executable,"-m","pip","install","-q",
    "tensorly","tensorly-torch","einops","timm","scipy"
],timeout=1200)
if rc:
    raise RuntimeError(o[-5000:])

sys.path.insert(0,str(SRC/"pdebench"))
sys.path.insert(0,str(SRC/"neuraloperator"))
TRANS_BENCH=SRC/"transolver"/"PDE-Solving-StandardBenchmark"
sys.path.insert(0,str(TRANS_BENCH))

from pdebench.models.fno.fno import FNO1d as PDE_FNO1d, FNO2d as PDE_FNO2d
from pdebench.models.unet.unet import UNet1d as PDE_UNet1d, UNet2d as PDE_UNet2d
from neuralop.models import FNO as NeuralOpFNO
from model.Transolver_Irregular_Mesh import Model as TransolverIrregular
from model.Transolver_Structured_Mesh_2D import Model as Transolver2D

print("OFFICIAL IMPORTS: PASS")
REPORT["source_checkout"]=checkout



# 3. Eight-task official PDEBench data panel

本轮数据文件固定在 PDEBench 官方归档 commit 对应的 DaRUS 清单中：

- Advection beta0.4
- Burgers nu0.01
- Diffusion-Sorption
- 1D Reaction-Diffusion nu1 rho2
- 1D Compressible CFD periodic
- 2D Diffusion-Reaction
- 2D Shallow Water
- 2D Compressible CFD 128 periodic

### 数据解析原则

脚本支持三类 PDEBench HDF5 schema：

1. `tensor` — Advection/Burgers/1D reaction-diffusion/diff-sorp；
2. grouped `*/data` — 2D reaction-diffusion/SWE；
3. CFD fields — `density`, `pressure`, `Vx`, optional `Vy`。

CFD 的物理变量直接堆叠为 channel，
不会删去 pressure/velocity 等字段。

### Compact cache

每个 task 只截取正式协议需要的：
- selected trajectories；
- selected history/target windows；
- per-channel normalization。

compact cache 写入：

```text
/content/CIDM_A2_BroadDynamics_Cache
```

并上传 HF Dataset。
下一次 Colab 优先恢复 cache，避免重新下载/解析大文件。


In [ ]:

# ============================================================
# 3.1 A2 HF DATASET persistent runtime
# ============================================================
from datetime import datetime, timezone

HF_DATASET_REPO_ID=os.getenv(
    "CIDM_HF_DATASET_REPO",
    "wuff-mann/CIDM-v1"
).strip()
HF_REPO_TYPE="dataset"
HF_PROGRESS_PREFIX="A2_BroadDynamics_500K_3Seed"
HF_CHECKPOINT_EVERY=int(os.getenv("CIDM_HF_CHECKPOINT_EVERY","500"))
HF_MAX_RETRIES=int(os.getenv("CIDM_HF_MAX_RETRIES","4"))
HF_ALLOW_LOCAL_ONLY=os.getenv("CIDM_HF_ALLOW_LOCAL_ONLY","0")=="1"

try:
    from google.colab import userdata
    HF_TOKEN_RESOLVED=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN_RESOLVED=os.getenv("HF_TOKEN")

HF_ENABLED=False
HF_API=None
HF_REMOTE_FILES=[]

def _utc_now():
    return datetime.now(timezone.utc).isoformat()

def _retry_call(fn,*args,**kwargs):
    last=None
    for attempt in range(1,HF_MAX_RETRIES+1):
        try:
            return fn(*args,**kwargs)
        except Exception as e:
            last=e
            if attempt==HF_MAX_RETRIES:
                break
            wait=min(30,2**attempt)
            print(f"[HF retry {attempt}] {type(e).__name__}: {e}; sleep {wait}s")
            time.sleep(wait)
    raise last

def _a2_remote_to_local(remote_path):
    rp=str(remote_path).replace("\\","/").lstrip("/")
    name=Path(rp).name

    pref=HF_PROGRESS_PREFIX+"/"
    if rp.startswith(pref):
        rel=rp[len(pref):]
        if rel.startswith("cache/"):
            return CACHE_ROOT/rel[len("cache/"):],100,"cache"
        return OUT/rel,100,"standard"

    # basename-compatible restore if an older upload layout is used.
    if name.startswith("A2__") and name.endswith("_result.json"):
        return OUT/name,80,"result"
    if name.startswith("A2__") and name.endswith("_history.csv"):
        return OUT/name,80,"history"
    if name.startswith("A2__") and name.endswith("_latest.pt"):
        return CKPT/name,80,"checkpoint"

    return None,-1,None

def _smart_copy_a2(src,dst):
    src=Path(src); dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)

    if not dst.exists():
        shutil.copy2(src,dst)
        return "copied"

    if dst.name.endswith("_result.json"):
        try:
            l=json.loads(dst.read_text(encoding="utf-8"))
            r=json.loads(src.read_text(encoding="utf-8"))
            if l.get("status")=="COMPLETED":
                return "kept_local_completed"
            if r.get("status")=="COMPLETED":
                shutil.copy2(src,dst); return "remote_completed"
        except Exception:
            pass
        return "kept_local"

    if dst.name.endswith("_latest.pt"):
        try:
            l=torch.load(dst,map_location="cpu",weights_only=False)
            r=torch.load(src,map_location="cpu",weights_only=False)
            ls=int(l.get("step",-1)); rs=int(r.get("step",-1))
            if rs>ls:
                shutil.copy2(src,dst)
                return f"remote_step_{rs}"
            return f"kept_local_step_{ls}"
        except Exception:
            return "kept_local"

    if dst.name.endswith("_history.csv"):
        try:
            l=pd.read_csv(dst); r=pd.read_csv(src)
            ls=int(l.step.max()) if len(l) else -1
            rs=int(r.step.max()) if len(r) else -1
            if rs>ls:
                shutil.copy2(src,dst)
                return f"remote_history_{rs}"
            return f"kept_local_history_{ls}"
        except Exception:
            return "kept_local"

    # Cache: existing verified local file wins.
    return "kept_local"

def scan_a2_progress():
    rows=[]
    for p in sorted(OUT.glob("A2__*_result.json")):
        try:
            x=json.loads(p.read_text(encoding="utf-8"))
            rows.append(x)
        except Exception:
            pass
    completed=[x for x in rows if x.get("status")=="COMPLETED"]
    checkpoints=sorted(CKPT.glob("A2__*_latest.pt"))
    return {
        "total_expected":len(SEEDS)*8*len(MODELS),
        "result_files":len(rows),
        "completed":len(completed),
        "latest_checkpoints":len(checkpoints),
    }

def write_a2_manifest(current_run=None,current_step=None,event=None):
    state=scan_a2_progress()
    state.update({
        "experiment":"CIDM Foundation A2 Broad Dynamics Benchmark",
        "repo_id":HF_DATASET_REPO_ID,
        "repo_type":"dataset",
        "prefix":HF_PROGRESS_PREFIX,
        "updated_at_utc":_utc_now(),
        "current_run":current_run,
        "current_step":current_step,
        "event":event,
    })
    p=OUT/"A2_progress_manifest.json"
    p.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding="utf-8")

    rows=[]
    for q in sorted(OUT.glob("A2__*_result.json")):
        try:
            rows.append(json.loads(q.read_text(encoding="utf-8")))
        except Exception:
            pass
    if rows:
        pd.DataFrame(rows).to_csv(OUT/"A2_run_results_partial.csv",index=False)
    return p,state

def _repo_path(local_path):
    p=Path(local_path).resolve()
    if CACHE_ROOT.resolve() in p.parents:
        rel=p.relative_to(CACHE_ROOT.resolve()).as_posix()
        return f"{HF_PROGRESS_PREFIX}/cache/{rel}"
    rel=p.relative_to(OUT.resolve()).as_posix()
    return f"{HF_PROGRESS_PREFIX}/{rel}"

def _hf_commit(local_files,message):
    if not HF_ENABLED:
        return False
    from huggingface_hub import CommitOperationAdd
    ops=[]; seen=set()
    for p in local_files:
        p=Path(p)
        if not p.exists() or not p.is_file():
            continue
        rp=_repo_path(p)
        if rp in seen:
            continue
        seen.add(rp)
        ops.append(CommitOperationAdd(path_in_repo=rp,path_or_fileobj=str(p)))
    if not ops:
        return True
    _retry_call(
        HF_API.create_commit,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        operations=ops,
        commit_message=message,
    )
    return True

def hf_sync_a2_checkpoint(path,run_id,step):
    if not HF_ENABLED:
        return
    man,_=write_a2_manifest(run_id,step,"MID_RUN_CHECKPOINT")
    try:
        _hf_commit(
            [path,man],
            f"[A2] checkpoint {run_id} step={step}"
        )
        print(f"[HF A2 CHECKPOINT] {run_id} step={step}")
    except Exception as e:
        print("[HF A2 checkpoint warning]",e)

def hf_sync_a2_finished(run_id,result_path,history_path,latest_path):
    if not HF_ENABLED:
        return
    try:
        row=json.loads(Path(result_path).read_text(encoding="utf-8"))
        status=row.get("status","UNKNOWN")
    except Exception:
        status="UNKNOWN"

    man,state=write_a2_manifest(run_id,None,f"RUN_{status}")
    files=[
        result_path,
        history_path,
        man,
        OUT/"A2_run_results_partial.csv",
    ]

    # first seed prediction artifact
    try:
        parts=run_id.split("__")
        model=parts[1]; task=parts[2]; seed=int(parts[3][1:])
        if seed==SEEDS[0]:
            q=PRED/f"{task}__{model}.pt"
            if q.exists():
                files.append(q)
    except Exception:
        pass

    if status!="COMPLETED" and Path(latest_path).exists():
        files.append(latest_path)

    try:
        _hf_commit(
            files,
            f"[A2] {status} {run_id} | completed={state['completed']}/{state['total_expected']}"
        )
        print(
            f"[HF A2 RUN SYNC] {run_id}: {status}; "
            f"{state['completed']}/{state['total_expected']}"
        )
        if status=="COMPLETED":
            try:
                HF_API.delete_file(
                    path_in_repo=_repo_path(latest_path),
                    repo_id=HF_DATASET_REPO_ID,
                    repo_type="dataset",
                    commit_message=f"[A2] cleanup {run_id} checkpoint",
                )
            except Exception:
                pass
    except Exception as e:
        print("[HF A2 run sync warning]",e)

def hf_sync_a2_cache(cache_path):
    if not HF_ENABLED:
        return
    marker=Path(str(cache_path)+".hf_uploaded")
    if marker.exists():
        return
    try:
        _hf_commit([cache_path],f"[A2] compact cache {Path(cache_path).name}")
        marker.write_text(_utc_now(),encoding="utf-8")
        print("[HF A2 CACHE]",Path(cache_path).name)
    except Exception as e:
        print("[HF A2 cache warning]",e)

def restore_a2_from_hf():
    global HF_REMOTE_FILES
    HF_REMOTE_FILES=_retry_call(
        HF_API.list_repo_files,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        revision="main",
    )
    candidates=[]
    for rf in HF_REMOTE_FILES:
        dst,prio,kind=_a2_remote_to_local(rf)
        if dst is not None:
            candidates.append((prio,rf,dst,kind))

    # Detect prior run progress before restore.
    remote_progress=[
        x for x in candidates
        if x[3] in {"result","history","checkpoint","standard"}
        and (
            Path(x[1]).name.startswith("A2__")
            or "A2_progress_manifest" in x[1]
        )
    ]

    if not candidates:
        print("[HF A2 RESTORE] no prior A2 files; fresh experiment.")
        return scan_a2_progress()

    from huggingface_hub import hf_hub_download
    restore_root=Path("/content/_cidm_a2_hf_restore")
    restore_root.mkdir(parents=True,exist_ok=True)

    candidates.sort(key=lambda x:x[0])
    for prio,rf,dst,kind in candidates:
        try:
            local=_retry_call(
                hf_hub_download,
                repo_id=HF_DATASET_REPO_ID,
                filename=rf,
                repo_type="dataset",
                token=HF_TOKEN_RESOLVED,
                revision="main",
                local_dir=str(restore_root),
            )
            act=_smart_copy_a2(Path(local),dst)
            if kind in {"result","checkpoint"} or Path(rf).name.startswith("A2__"):
                print(f"  [A2 restore] {rf} -> {dst.name} ({act})")
        except Exception as e:
            print(f"  [A2 restore warning] {rf}: {e}")

    state=scan_a2_progress()
    print(
        f"[HF A2 RESTORE COMPLETE] "
        f"completed={state['completed']}/{state['total_expected']} "
        f"checkpoints={state['latest_checkpoints']}"
    )

    # Safety gate: remote run progress existed but nothing became resumable.
    if remote_progress and state["result_files"]==0 and state["latest_checkpoints"]==0:
        raise RuntimeError(
            "HF contains prior A2 run progress but zero local progress was reconstructed. "
            "Training blocked to avoid accidental restart."
        )
    return state

# initialize Dataset repo
if HF_TOKEN_RESOLVED:
    try:
        subprocess.run(
            [sys.executable,"-m","pip","install","-q","-U","huggingface_hub"],
            check=True,
        )
        from huggingface_hub import HfApi
        HF_API=HfApi(token=HF_TOKEN_RESOLVED)
        who=_retry_call(HF_API.whoami)
        print("[HF AUTH]",who.get("name"))
        _retry_call(
            HF_API.dataset_info,
            repo_id=HF_DATASET_REPO_ID,
            revision="main",
        )
        HF_ENABLED=True
        print(
            f"[HF DATASET VERIFIED] "
            f"https://huggingface.co/datasets/{HF_DATASET_REPO_ID}"
        )
        restore_a2_from_hf()
    except Exception as e:
        HF_ENABLED=False
        if HF_ALLOW_LOCAL_ONLY:
            print("[HF A2 WARNING]",repr(e))
        else:
            raise RuntimeError(
                f"A2 HF Dataset initialization/restore failed: {repr(e)}"
            ) from e
else:
    if not HF_ALLOW_LOCAL_ONLY:
        raise RuntimeError("HF_TOKEN unavailable; A2 persistent runtime requires it.")
    print("[HF A2] local-only mode.")


In [ ]:

# ============================================================
# 3. Official A2 data / deterministic compact caches
# ============================================================
TASKS={
    "advection_1d":{
        "family":"transport",
        "filename":"1D_Advection_Sols_beta0.4.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/255674",
        "md5":"d595bbfd2c659df995a93cd40d6ea568",
        "spatial_stride":4,
    },
    "burgers_1d":{
        "family":"nonlinear_transport",
        "filename":"1D_Burgers_Sols_Nu0.01.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/281363",
        "md5":"e6d9a4f62baf9a29121a816b919e2770",
        "spatial_stride":4,
    },
    "diff_sorp_1d":{
        "family":"diffusion_sorption",
        "filename":"1D_diff-sorp_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133020",
        "md5":"9d466d1213065619d087319e16d9a938",
        "spatial_stride":4,
    },
    "reacdiff_1d":{
        "family":"reaction_diffusion",
        "filename":"ReacDiff_Nu1.0_Rho2.0.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133183",
        "md5":"112c01a76447162bd67c8c1073f58ca2",
        "spatial_stride":4,
    },
    "cfd_1d":{
        "family":"compressible_cfd_1d",
        "filename":"1D_CFD_Rand_Eta0.01_Zeta0.01_periodic_Train.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/164672",
        "md5":"2fdd00138f1d7abc794fb953021a9f43",
        "spatial_stride":4,
    },
    "reacdiff_2d":{
        "family":"reaction_diffusion_2d",
        "filename":"2D_diff-react_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133017",
        "md5":"b8d0b86064193195ddc30c33be5dc949",
        "spatial_stride":1,
    },
    "swe_2d":{
        "family":"shallow_water",
        "filename":"2D_rdb_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133021",
        "md5":"75d838c47aa410694bdc912ea7f22282",
        "spatial_stride":1,
    },
    "cfd_2d":{
        "family":"compressible_cfd_2d",
        "filename":"2D_CFD_Rand_M0.1_Eta0.01_Zeta0.01_periodic_128_Train.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/164687",
        "md5":"5b21dcccaef4d2145ca579a71153c580",
        "spatial_stride":2,
    },
}

def resume_download(url,path,expected_md5,chunk=8*1024*1024):
    path=Path(path)
    if path.exists() and md5_file(path)==expected_md5:
        print("[verified data]",path.name)
        return
    if path.exists():
        path.unlink()

    part=path.with_suffix(path.suffix+".part")
    pos=part.stat().st_size if part.exists() else 0
    headers={"Range":f"bytes={pos}-"} if pos else {}

    with requests.get(url,stream=True,headers=headers,timeout=(30,300)) as r:
        if pos and r.status_code==200:
            part.unlink(missing_ok=True)
            pos=0
        r.raise_for_status()
        mode="ab" if pos and r.status_code==206 else "wb"
        with open(part,mode) as f:
            for b in r.iter_content(chunk_size=chunk):
                if b:
                    f.write(b)

    part.replace(path)
    got=md5_file(path)
    if got!=expected_md5:
        raise RuntimeError(f"MD5 mismatch {path.name}: {got}")

def inspect_h5(path):
    with h5py.File(path,"r") as f:
        if "tensor" in f:
            return {
                "schema":"tensor",
                "n":f["tensor"].shape[0],
                "keys":list(f.keys()),
            }

        if all(k in f for k in ["density","pressure","Vx"]):
            return {
                "schema":"cfd",
                "n":f["density"].shape[0],
                "keys":list(f.keys()),
            }

        groups=[
            k for k in sorted(f.keys())
            if isinstance(f[k],h5py.Group) and "data" in f[k]
        ]
        if groups:
            return {
                "schema":"grouped",
                "n":len(groups),
                "keys":groups[:5],
            }

        raise RuntimeError(
            f"Unsupported PDEBench HDF5 schema for {path}; keys={list(f.keys())[:30]}"
        )

def _standardize_tensor(a):
    """
    Return [T,C,*S] from common PDEBench tensor-like arrays.
    """
    a=np.asarray(a,dtype=np.float32)

    if a.ndim==2:          # [T,X]
        return a[:,None,:]

    if a.ndim==3:
        # Usually [T,X,Y] for scalar 2D, or [T,X,C] uncommon.
        return a[:,None,:,:]

    if a.ndim==4:
        # Assume PDEBench grouped convention [T,X,Y,C] when last axis is small.
        if a.shape[-1] <= 8:
            return np.moveaxis(a,-1,1)
        # Otherwise interpret [T,C,X,Y].
        if a.shape[1] <= 8:
            return a

    raise RuntimeError(f"Cannot standardize array shape {a.shape}")

def read_traj(task,path,idx):
    stride=int(TASKS[task].get("spatial_stride",1))

    with h5py.File(path,"r") as f:
        if "tensor" in f:
            a=np.asarray(f["tensor"][idx],dtype=np.float32)

            # Tensor files in selected 1D tasks are [T,X].
            if a.ndim==2:
                a=a[:,None,:]
            elif a.ndim==3:
                # If selected task is explicitly 2D, single-channel 2D.
                if task.endswith("_2d"):
                    a=a[:,None,:,:]
                else:
                    # 1D multi-channel fallback [T,X,C]
                    if a.shape[-1] <= 8:
                        a=np.moveaxis(a,-1,1)
                    else:
                        a=a[:,None,:,:]
            elif a.ndim==4:
                if a.shape[-1] <= 8:
                    a=np.moveaxis(a,-1,1)
                elif a.shape[1] <= 8:
                    pass
                else:
                    raise RuntimeError((task,a.shape))
            else:
                raise RuntimeError((task,a.shape))

        elif all(k in f for k in ["density","pressure","Vx"]):
            fields=[
                np.asarray(f["density"][idx],dtype=np.float32),
                np.asarray(f["pressure"][idx],dtype=np.float32),
                np.asarray(f["Vx"][idx],dtype=np.float32),
            ]
            if "Vy" in f:
                fields.append(np.asarray(f["Vy"][idx],dtype=np.float32))
            a=np.stack(fields,axis=1)  # [T,C,*S]

        else:
            groups=[
                k for k in sorted(f.keys())
                if isinstance(f[k],h5py.Group) and "data" in f[k]
            ]
            if not groups:
                raise RuntimeError(f"No grouped data in {path}")
            a=np.asarray(f[groups[idx]]["data"],dtype=np.float32)
            if a.ndim==4:
                a=np.moveaxis(a,-1,1)   # [T,X,Y,C] -> [T,C,X,Y]
            elif a.ndim==3:
                a=a[:,None,:,:]
            elif a.ndim==2:
                a=a[:,None,:]
            else:
                raise RuntimeError((task,a.shape))

    # Spatial subsampling is deterministic and identical for all models.
    if stride>1:
        if a.ndim==3:
            a=a[...,::stride]
        elif a.ndim==4:
            a=a[...,::stride,::stride]

    if not np.isfinite(a).all():
        raise FloatingPointError(f"{task} trajectory {idx} contains non-finite values")
    return torch.from_numpy(np.ascontiguousarray(a))

def even_pick(pool,n):
    pool=np.asarray(pool,dtype=int)
    n=min(n,len(pool))
    if n<=0:
        return []
    return pool[np.linspace(0,len(pool)-1,n,dtype=int)].tolist()

def split_ids(N):
    nt=max(1,int(.10*N))
    test_pool=np.arange(0,nt)
    trainval=np.arange(nt,N)
    nv=max(1,int(.10*N))
    val_pool=trainval[-nv:]
    train_pool=trainval[:-nv]
    return {
        "train":even_pick(train_pool,N_TRAIN_TRAJ),
        "val":even_pick(val_pool,N_VAL_TRAJ),
        "test":even_pick(test_pool,N_TEST_TRAJ),
    }

def cache_path(task,m):
    return CACHE_ROOT/(
        f"{task}_L{HISTORY}_K{HORIZON}"
        f"_tr{N_TRAIN_TRAJ}_va{N_VAL_TRAJ}_te{N_TEST_TRAJ}"
        f"_w{WINDOWS_PER_TRAJ}_s{m['spatial_stride']}.pt"
    )

def validate_cache(c,m):
    return (
        c.get("source_md5")==m["md5"]
        and c.get("history")==HISTORY
        and c.get("horizon")==HORIZON
        and c.get("spatial_stride")==m["spatial_stride"]
    )

def build_cache(task,m):
    cp=cache_path(task,m)

    if cp.exists():
        try:
            c=torch.load(cp,map_location="cpu",weights_only=False)
            if validate_cache(c,m):
                print("[reuse compact cache]",cp.name)
                return c
        except Exception:
            cp.unlink(missing_ok=True)

    # Compact cache not restored from HF => obtain official raw file.
    raw=DATA_ROOT/m["filename"]
    resume_download(m["url"],raw,m["md5"])

    info=inspect_h5(raw)
    ids=split_ids(info["n"])
    out={
        "task":task,
        "family":m["family"],
        "source_filename":m["filename"],
        "source_md5":m["md5"],
        "schema":info["schema"],
        "history":HISTORY,
        "horizon":HORIZON,
        "spatial_stride":m["spatial_stride"],
        "selected_trajectory_ids":ids,
        "splits":{},
    }

    for split,idxs in ids.items():
        HH=[]; YY=[]; WM=[]
        for idx in idxs:
            tr=read_traj(task,raw,idx)
            mx=tr.shape[0]-HISTORY-HORIZON
            if mx<0:
                raise RuntimeError(
                    f"{task}: T={tr.shape[0]} shorter than HISTORY+HORIZON"
                )
            starts=np.unique(
                np.linspace(0,mx,WINDOWS_PER_TRAJ,dtype=int)
            )
            for st in starts:
                HH.append(tr[st:st+HISTORY])
                YY.append(tr[st+HISTORY:st+HISTORY+HORIZON])
                WM.append((int(idx),int(st)))

        out["splits"][split]={
            "history":torch.stack(HH).contiguous(),
            "target":torch.stack(YY).contiguous(),
            "window_meta":WM,
        }

    # Per-channel train statistics over history + target.
    h=out["splits"]["train"]["history"].double()
    y=out["splits"]["train"]["target"].double()
    C=h.shape[2]

    sum_c=torch.zeros(C,dtype=torch.float64)
    sq_c=torch.zeros(C,dtype=torch.float64)
    count=0
    for q in [h,y]:
        reduce_dims=(0,1)+tuple(range(3,q.ndim))
        sum_c+=q.sum(dim=reduce_dims)
        sq_c+=(q*q).sum(dim=reduce_dims)
        n=1
        for d in reduce_dims:
            n*=q.shape[d]
        count+=n

    mean=sum_c/count
    var=(sq_c/count-mean*mean).clamp_min(1e-12)
    std=torch.sqrt(var)
    out["norm"]={"mean":mean.float(),"std":std.float()}

    torch.save(out,cp)
    print("[built compact cache]",cp)

    # Immediately persist compact cache for future Colab sessions.
    hf_sync_a2_cache(cp)
    return out

CACHES={}
for task,m in TASKS.items():
    CACHES[task]=build_cache(task,m)

class ChannelNormalizer:
    def __init__(self,mean,std):
        self.mean=mean.float()
        self.std=std.float().clamp_min(1e-6)

    def norm_history(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return (
            x-self.mean.view(*shape).to(x.device)
        )/self.std.view(*shape).to(x.device)

    def norm_target(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return (
            x-self.mean.view(*shape).to(x.device)
        )/self.std.view(*shape).to(x.device)

    def denorm_target(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return (
            x*self.std.view(*shape).to(x.device)
            +self.mean.view(*shape).to(x.device)
        )

NORMALIZERS={
    t:ChannelNormalizer(c["norm"]["mean"],c["norm"]["std"])
    for t,c in CACHES.items()
}

class FrozenWindows(Dataset):
    def __init__(self,c,split,norm):
        self.h=c["splits"][split]["history"].float()
        self.y=c["splits"][split]["target"].float()
        self.norm=norm

    def __len__(self):
        return len(self.h)

    def __getitem__(self,i):
        h=self.h[i]
        y=self.y[i]
        mh=self.norm.mean.view(1,-1,*([1]*(h.ndim-2)))
        sh=self.norm.std.view(1,-1,*([1]*(h.ndim-2)))
        return (h-mh)/sh,(y-mh)/sh

def loaders_for(task,batch,seed):
    c=CACHES[task]
    norm=NORMALIZERS[task]
    g=torch.Generator().manual_seed(seed)
    return (
        DataLoader(
            FrozenWindows(c,"train",norm),
            batch_size=batch,
            shuffle=True,
            generator=g,
            num_workers=0,
            drop_last=True,
            pin_memory=True,
        ),
        DataLoader(
            FrozenWindows(c,"val",norm),
            batch_size=batch,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        ),
        DataLoader(
            FrozenWindows(c,"test",norm),
            batch_size=batch,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        ),
    )

def task_spec(task):
    h=CACHES[task]["splits"]["train"]["history"]
    return h.ndim-3,h.shape[2],tuple(h.shape[3:])

manifest=[]
for t,m in TASKS.items():
    d,c,s=task_spec(t)
    cc=CACHES[t]
    manifest.append({
        "task":t,
        "family":m["family"],
        "dim":d,
        "channels":c,
        "spatial":list(s),
        "stride":m["spatial_stride"],
        "schema":cc["schema"],
        "train_windows":len(cc["splits"]["train"]["history"]),
        "val_windows":len(cc["splits"]["val"]["history"]),
        "test_windows":len(cc["splits"]["test"]["history"]),
        "norm_mean":cc["norm"]["mean"].tolist(),
        "norm_std":cc["norm"]["std"].tolist(),
    })

manifest_df=pd.DataFrame(manifest)
manifest_df.to_csv(OUT/"A2_data_manifest.csv",index=False)
display(manifest_df)

# global finite audit
for task,c in CACHES.items():
    for split in ["train","val","test"]:
        for key in ["history","target"]:
            x=c["splits"][split][key]
            assert torch.isfinite(x).all(),(task,split,key)

print("A2 DATA/CACHE AUDIT: PASS")
REPORT["data_manifest"]=manifest



# 4. Frozen / official model wrappers

继续沿用 A1-R2-R1 的同一套 wrapper 原则：

- CIDM Core：冻结 canonical implementation；
- CIDM-Complete：冻结 archived executable topology；
- PDEBench FNO：官方源码；
- NeuralOperator FNO：官方源码；
- Transolver：官方源码。

Wrapper 只负责：
- tensor layout
- regular-grid coordinates
- autoregressive history update
- Complete 的 1D `H=1` lift/squeeze

不增加任何任务特定 CNN 或额外未来信息。


In [ ]:

# ============================================================
# 4. Official wrappers
# ============================================================
def normalized_grid(spatial,device):
    if len(spatial)==1:
        return torch.linspace(0,1,spatial[0],device=device)[:,None]
    x=torch.linspace(0,1,spatial[0],device=device)
    y=torch.linspace(0,1,spatial[1],device=device)
    X,Y=torch.meshgrid(x,y,indexing="ij")
    return torch.stack([X,Y],-1)

class CIDMWrapper(nn.Module):
    def __init__(self,dim,c,L,K,D,H):
        super().__init__()
        self.model=build_cidm_foundation_core(
            spatial_dim=dim,channels=c,hist_len=L,horizon=K,
            latent_dim=D,hidden=H
        )
    def rollout(self,h,K):
        return self.model(h,horizon=K)["pred"]


class CIDMCompleteWrapper(nn.Module):
    """
    Exact archived CIDM_BER topology used as CIDM-Complete.
    For 1D tasks we lift X -> [1,X] and use the same archived Conv2D model.
    No external forcing/port is supplied in the PDEBench headline panel.
    """
    def __init__(self,dim,c,L,K,D,H,modes1,modes2):
        super().__init__()
        self.dim=dim
        self.config=CIDMCompleteConfig(
            channels=c,
            hist_len=L,
            horizon=K,
            latent_dim=D,
            hidden=H,
            port_channels=1,
            use_phase_encoder=True,
            use_time_encoder=True,
            use_hidden_reservoir=True,
            use_flux=True,
            use_reverse_diagnostics=True,
            soc_modes1=modes1,
            soc_modes2=modes2,
            ber_add_scale=0.05,
            ber_enabled=True,
            soc_enabled=True,
        )
        self.model=CIDMCompleteArchived(self.config)

    def _lift(self,h):
        return h.unsqueeze(-2) if self.dim==1 else h

    def rollout(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        p=out["pred"]
        return p.squeeze(-2) if self.dim==1 else p

    @torch.no_grad()
    def architecture_diagnostics(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        d=out["diagnostics"]
        return {
            "latent_energy_drift":float(d["energy_drift"].detach()),
            "soc_strength":float(d["soc_strength"].detach()),
            "ber_strength":float(d["ber_strength"].detach()),
        }


class PDEBenchFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        if dim==1:
            self.model=PDE_FNO1d(c,modes=modes,width=width,initial_step=L)
        else:
            self.model=PDE_FNO2d(c,modes1=modes,modes2=modes,width=width,initial_step=L)
    def one(self,h):
        B,L,C,*S=h.shape
        if self.dim==1:
            inp=h.permute(0,3,1,2).reshape(B,S[0],L*C)
        else:
            inp=h.permute(0,3,4,1,2).reshape(B,S[0],S[1],L*C)
        grid=normalized_grid(S,h.device)[None].expand(B,*([-1]*self.dim),-1)
        y=self.model(inp,grid).squeeze(-2)
        if self.dim==1: return y.permute(0,2,1).contiguous()
        return y.permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class PDEBenchUNetWrapper(nn.Module):
    def __init__(self,dim,c,L,features):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=(PDE_UNet1d if dim==1 else PDE_UNet2d)(
            in_channels=c*L,out_channels=c,init_features=features
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class NeuralOpFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=NeuralOpFNO(
            n_modes=tuple([modes]*dim),
            in_channels=c*L,out_channels=c,
            hidden_channels=width,n_layers=4,
            positional_embedding="grid",
            fno_block_precision="full",
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class TransolverWrapper(nn.Module):
    def __init__(self,dim,c,L,S,n_hidden,n_layers,n_head,slice_num):
        super().__init__(); self.dim=dim; self.c=c; self.L=L; self.S=S
        kw=dict(
            space_dim=dim,n_layers=n_layers,n_hidden=n_hidden,dropout=0.0,
            n_head=n_head,Time_Input=False,act="gelu",mlp_ratio=1,
            fun_dim=c*L,out_dim=c,slice_num=slice_num,unified_pos=False
        )
        if dim==1:
            self.model=TransolverIrregular(**kw)
        else:
            self.model=Transolver2D(**kw,H=S[0],W=S[1])
    def one(self,h):
        B,L,C,*S=h.shape
        g=normalized_grid(S,h.device).reshape(-1,self.dim)[None].expand(B,-1,-1)
        if self.dim==1:
            fx=h.permute(0,3,1,2).reshape(B,-1,L*C)
        else:
            fx=h.permute(0,3,4,1,2).reshape(B,-1,L*C)
        y=self.model(g,fx)
        if self.dim==1:
            return y.reshape(B,S[0],C).permute(0,2,1).contiguous()
        return y.reshape(B,S[0],S[1],C).permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)



# 5. Per-task 0.5M parameter-only matching

A2 新任务拥有不同 channel 数和空间尺寸，因此不能直接复用三任务旧配置。

本轮只根据：

\[
|\#Params-500K|
\]

搜索官方/公开 constructor knobs。

参数匹配与预测性能完全解耦：
- 不训练；
- 不看 validation；
- 不根据结果调整。

所有模型要求：

\[
500K\pm5\%
\]

否则正式训练 Gate 阻断。


In [ ]:

# ============================================================
# 5. A2 automatic parameter-only matching
# ============================================================
def build_by_config(task,name,cfg):
    dim,c,S=task_spec(task)

    if name=="CIDM-Foundation-v1":
        return CIDMWrapper(
            dim,c,HISTORY,HORIZON,
            cfg["latent_dim"],cfg["hidden"]
        )

    if name=="CIDM-Complete":
        return CIDMCompleteWrapper(
            dim,c,HISTORY,HORIZON,
            cfg["latent_dim"],cfg["hidden"],
            cfg["soc_modes1"],cfg["soc_modes2"]
        )

    if name=="PDEBench-FNO":
        return PDEBenchFNOWrapper(
            dim,c,HISTORY,cfg["width"],cfg["modes"]
        )

    if name=="NeuralOperator-FNO":
        return NeuralOpFNOWrapper(
            dim,c,HISTORY,cfg["width"],cfg["modes"]
        )

    if name=="Transolver":
        return TransolverWrapper(
            dim,c,HISTORY,S,
            cfg["n_hidden"],cfg["n_layers"],
            cfg["n_head"],cfg["slice_num"]
        )

    raise KeyError(name)

def search_cidm_core(task):
    dim,c,S=task_spec(task)
    cands=[]
    for D in range(16,145,4):
        hs=sorted(set([
            max(16,D),
            max(16,int(round(1.5*D/8)*8)),
            max(16,2*D),
            max(16,int(round(2.5*D/8)*8)),
            max(16,3*D),
        ]))
        for H in hs:
            m=CIDMWrapper(dim,c,HISTORY,HORIZON,D,H)
            p=count_params(m)
            cands.append((abs(p-TARGET_PARAMS),p,{
                "latent_dim":D,"hidden":H
            }))
            del m
    return min(cands,key=lambda x:x[0])

def complete_param_formula(c,D,H,m1,m2,L=4):
    # Exact frozen cidm_architectures.py::CIDM_BER count.
    space=9*c*D + 7*D + 10*D*D
    timec=9*c*L*D + 7*D + 10*D*D
    phase=11*D*D + 3*c*D + 10*D
    fuse=4*D*D + 2*D

    skew=D*D
    flux=3*D*H + H + D
    conv_dd=10*D*H + 9*H*H + 6*H + D
    hidden=conv_dd + 1
    local=conv_dd
    core=skew+flux+hidden+local+1

    decoder=9*D*H + 9*H*H + 6*H + H*c + c
    spectral_decoder=D*D + D + D*c + c + 1

    soc=4*D*D*m1*m2 + (D*D+3*D) + (2*D*H+H+D) + 1
    M=max(8,H//2)
    ber=(27*D*H + 9*H*H + 15*H + D) + (18*D*M + 11*M + 1) + 1
    return int(
        space+timec+phase+fuse+core+decoder+spectral_decoder+soc+ber
    )

def search_cidm_complete(task):
    dim,c,S=task_spec(task)
    m1s=[1] if dim==1 else [2,4,6,8]
    max_mode=max(4,min(14,min(S)//4))
    m2s=sorted(set([4,6,8,10,12,14]))
    m2s=[m for m in m2s if m<=max(4,max_mode)]

    cands=[]
    for D in range(8,65,2):
        for H in range(16,129,4):
            for m1 in m1s:
                for m2 in m2s:
                    p=complete_param_formula(c,D,H,m1,m2)
                    cands.append((
                        abs(p-TARGET_PARAMS),p,{
                            "latent_dim":D,
                            "hidden":H,
                            "soc_modes1":m1,
                            "soc_modes2":m2,
                        }
                    ))

    err,p,cfg=min(cands,key=lambda x:x[0])
    m=build_by_config(task,"CIDM-Complete",cfg)
    actual=count_params(m)
    del m
    if actual!=p:
        raise RuntimeError(
            f"Complete formula mismatch {task}: {p} vs {actual}"
        )
    return err,p,cfg

def candidate_modes(dim,S):
    lim=max(2,min(S)//2)
    if dim==1:
        raw=[4,6,8,10,12,16,20,24,32]
    else:
        raw=[4,6,8,10,12,14,16]
    return [m for m in raw if m<=lim] or [2]

def search_official_fno(task,name):
    dim,c,S=task_spec(task)
    modes_list=candidate_modes(dim,S)
    widths=range(8,129,2) if dim==1 else range(8,81,2)
    cands=[]

    # coarse bounded search: instantiate official class, count only.
    for modes in modes_list:
        for w in widths:
            cfg={"width":w,"modes":modes}
            try:
                m=build_by_config(task,name,cfg)
                p=count_params(m)
                cands.append((abs(p-TARGET_PARAMS),p,cfg))
                del m
            except Exception:
                continue
    if not cands:
        raise RuntimeError(f"No parameter candidate for {task} {name}")
    return min(cands,key=lambda x:x[0])

def search_transolver(task):
    dim,c,S=task_spec(task)
    slice_num=32 if dim==1 else 8
    cands=[]
    for layers in [4,5,6]:
        for h in range(32,193,4):
            if h%4:
                continue
            cfg={
                "n_hidden":h,
                "n_layers":layers,
                "n_head":4,
                "slice_num":slice_num,
            }
            try:
                m=build_by_config(task,"Transolver",cfg)
                p=count_params(m)
                cands.append((abs(p-TARGET_PARAMS),p,cfg))
                del m
            except Exception:
                continue
    if not cands:
        raise RuntimeError(f"No Transolver parameter candidate for {task}")
    return min(cands,key=lambda x:x[0])

SEARCHERS={
    "CIDM-Foundation-v1":search_cidm_core,
    "CIDM-Complete":search_cidm_complete,
    "PDEBench-FNO":lambda t:search_official_fno(t,"PDEBench-FNO"),
    "NeuralOperator-FNO":lambda t:search_official_fno(t,"NeuralOperator-FNO"),
    "Transolver":search_transolver,
}

MATCH={}
rows=[]
for task in TASKS:
    MATCH[task]={}
    for name in MODELS:
        err,p,cfg=SEARCHERS[name](task)
        rel=err/TARGET_PARAMS
        ok=rel<=PARAM_TOL
        MATCH[task][name]={
            "params":p,
            "rel_error":rel,
            "ok":ok,
            "config":cfg,
        }
        rows.append({
            "task":task,
            "model":name,
            "params":p,
            "rel_error":rel,
            "ok":ok,
            "config":json.dumps(cfg),
        })
        print(
            task,name,p,
            f"{100*rel:.2f}%",
            "PASS" if ok else "FAIL",
            cfg
        )
        gc.collect()

match_df=pd.DataFrame(rows)
match_df.to_csv(OUT/"A2_parameter_matching.csv",index=False)
display(match_df)

if not match_df["ok"].all():
    raise RuntimeError(
        "A2 parameter matching failed. Formal benchmark blocked."
    )

save_json(MATCH,OUT/"A2_parameter_matching.json")
REPORT["parameter_matching"]=rows



# 6. A2 pre-flight gate

正式训练前：

\[
8\ tasks \times 5\ models = 40
\]

个真实组合全部执行：

```text
real compact-cache batch
→ 6-step rollout
→ shape
→ finite
→ backward
→ gradient finite
→ optimizer step
→ checkpoint save/reload
→ inference finite
```

任意失败都停止，不允许“边跑 120 组边发现 adapter 有问题”。


In [ ]:

# ============================================================
# 6. Pre-flight
# ============================================================
PREFLIGHT=[]
for task in TASKS:
    dim,c,S=task_spec(task)
    batch=BATCH_1D if dim==1 else BATCH_2D
    tr,va,te=loaders_for(task,batch,DEV_SEED)
    h,y=next(iter(tr)); h,y=h.to(DEVICE),y.to(DEVICE)

    for name in MODELS:
        seed_all(DEV_SEED)
        cfg=MATCH[task][name]["config"]
        m=build_by_config(task,name,cfg).to(DEVICE).float()
        opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=WEIGHT_DECAY)
        ok=False; err=""
        try:
            m.train(); opt.zero_grad(set_to_none=True)
            p=m.rollout(h,HORIZON)
            if p.shape!=y.shape:
                raise RuntimeError(f"shape {p.shape} != {y.shape}")
            if not torch.isfinite(p).all():
                raise FloatingPointError("non-finite forward")
            loss=F.mse_loss(p,y)
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss")
            loss.backward()
            bad=[
                n for n,q in m.named_parameters()
                if q.grad is not None and not torch.isfinite(q.grad).all()
            ]
            if bad:
                raise FloatingPointError(f"non-finite gradients: {bad[:8]}")
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
            opt.step()
            if not all(torch.isfinite(q).all() for q in m.parameters()):
                raise FloatingPointError("non-finite params")

            sd=portable_state_dict(m)
            temp=CKPT/f"preflight__{task}__{name}.pt"
            torch.save(sd,temp)

            fresh=build_by_config(task,name,cfg).to(DEVICE).float()
            load_portable_state_dict(
                fresh,torch.load(temp,map_location="cpu",weights_only=False)
            )
            fresh.eval()
            with torch.no_grad():
                pp=fresh.rollout(h,HORIZON)
            if not torch.isfinite(pp).all():
                raise FloatingPointError("non-finite after reload")
            ok=True
        except Exception as e:
            err=repr(e)

        row={"task":task,"model":name,"pass":ok,"error":err}
        PREFLIGHT.append(row); print(row)
        del m
        if "fresh" in locals(): del fresh
        torch.cuda.empty_cache(); gc.collect()

pre_df=pd.DataFrame(PREFLIGHT)
pre_df.to_csv(OUT/"preflight.csv",index=False)
display(pre_df)
if not pre_df["pass"].all():
    raise RuntimeError("Pre-flight failed. Formal training is blocked.")
REPORT["preflight"]=PREFLIGHT



# 7. A2 metrics

沿用 A1-R2-R1 已修复的论文级指标：

预测：
- RMSE
- MAE
- Relative L2
- lead-wise RMSE
- rollout growth

频谱：
- low/mid/high band energy-fraction error
- low/mid/high log-spectrum RMSE

效率：
- actual updates
- GPU seconds
- latency
- throughput
- peak VRAM

CIDM 专属 diagnostics 继续只做解释，不进入 loss。


In [ ]:

# ============================================================
# 7. Publication metrics
# ============================================================
def _spectral_masks(shape,device):
    if len(shape)==1:
        f=torch.fft.rfftfreq(shape[0],device=device)
        q=f/f.max().clamp_min(1e-12)
    else:
        fy=torch.fft.fftfreq(shape[-2],device=device)[:,None]
        fx=torch.fft.rfftfreq(shape[-1],device=device)[None,:]
        r=torch.sqrt(fx*fx+fy*fy)
        q=r/r.max().clamp_min(1e-12)
    return {
        "low":q<=1/3,
        "mid":(q>1/3)&(q<=2/3),
        "high":q>2/3,
    }

def _publication_spectral_metrics(pred,target):
    # pred/target [B,K,C,*S], original physical units.
    dim=pred.ndim-3
    if dim==1:
        P=torch.fft.rfft(pred.float(),dim=-1,norm="ortho").abs()
        T=torch.fft.rfft(target.float(),dim=-1,norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-1],),pred.device)
    else:
        P=torch.fft.rfft2(pred.float(),dim=(-2,-1),norm="ortho").abs()
        T=torch.fft.rfft2(target.float(),dim=(-2,-1),norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-2],pred.shape[-1]),pred.device)

    P2=P.square()
    T2=T.square()
    reduce_freq=(-1,) if dim==1 else (-2,-1)
    totalP=P2.sum(dim=reduce_freq).clamp_min(1e-12)
    totalT=T2.sum(dim=reduce_freq).clamp_min(1e-12)

    out={}
    logP=torch.log1p(P)
    logT=torch.log1p(T)

    for band,m in masks.items():
        # Band energy fraction: stable even if true band energy is almost zero.
        bp=P2[...,m].sum(dim=-1)
        bt=T2[...,m].sum(dim=-1)
        fp=bp/totalP
        ft=bt/totalT
        out[f"band_fraction_abs_error_{band}"]=float((fp-ft).abs().mean())

        # Log-spectrum RMSE in that band.
        d=(logP[...,m]-logT[...,m]).square().mean().sqrt()
        out[f"logspec_rmse_{band}"]=float(d)

        # Store mean true fraction for interpretability.
        out[f"true_band_fraction_{band}"]=float(ft.mean())

    return out

@torch.no_grad()
def evaluate_model(model,loader,task,max_batches=None,save_examples=False,example_tag=None):
    model.eval()
    norm=NORMALIZERS[task]
    mse=[]; mae=[]; rel=[]; leads=[]; specs=[]
    n_samples=0
    first_example=None

    torch.cuda.synchronize()
    t0=time.perf_counter()

    for bi,(h,y_n) in enumerate(loader):
        if max_batches is not None and bi>=max_batches:
            break
        h=h.to(DEVICE,non_blocking=True)
        y_n=y_n.to(DEVICE,non_blocking=True)

        p_n=model.rollout(h,HORIZON)
        p=norm.denorm_target(p_n)
        y=norm.denorm_target(y_n)
        e=p-y

        mse.append(float((e**2).mean()))
        mae.append(float(e.abs().mean()))
        rel.append(float(
            torch.linalg.vector_norm(e.reshape(e.shape[0],-1),dim=1).div(
                torch.linalg.vector_norm(y.reshape(y.shape[0],-1),dim=1).clamp_min(1e-8)
            ).mean()
        ))
        leads.append(torch.sqrt((e**2).flatten(2).mean(-1)).mean(0).cpu())
        specs.append(_publication_spectral_metrics(p,y))
        n_samples += h.shape[0]

        if first_example is None:
            # save one deterministic example in original units
            hh=norm.denorm_target(h[:, -1:, ...]).squeeze(1)
            first_example={
                "last_input":hh[0].detach().cpu(),
                "target":y[0].detach().cpu(),
                "pred":p[0].detach().cpu(),
            }

    torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0

    le=torch.stack(leads).mean(0)
    out={
        "rmse":math.sqrt(np.mean(mse)),
        "mae":float(np.mean(mae)),
        "relative_l2":float(np.mean(rel)),
        "lead_rmse":[float(x) for x in le],
        "rollout_growth":float(le[-1]/le[0].clamp_min(1e-8)),
        "eval_throughput_windows_per_s":float(n_samples/max(elapsed,1e-9)),
    }

    for k in specs[0].keys():
        out[k]=float(np.mean([x[k] for x in specs]))

    if save_examples and first_example is not None and example_tag:
        torch.save(first_example,PRED/f"{example_tag}.pt")

    return out

@torch.no_grad()
def measure_latency(model,loader,repeats=50,warmup=15):
    model.eval()
    h,_=next(iter(loader))
    h=h[:1].to(DEVICE)

    for _ in range(warmup):
        _=model.rollout(h,HORIZON)
    torch.cuda.synchronize()

    ts=[]
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0=time.perf_counter()
        _=model.rollout(h,HORIZON)
        torch.cuda.synchronize()
        ts.append((time.perf_counter()-t0)*1000)

    return {
        "latency_ms_median":float(np.median(ts)),
        "latency_ms_mean":float(np.mean(ts)),
        "latency_ms_p95":float(np.percentile(ts,95)),
    }

@torch.no_grad()
def architecture_specific_diagnostics(model,loader,name,max_batches=4):
    if name not in {"CIDM-Foundation-v1","CIDM-Complete"}:
        return {}

    rows=[]
    for bi,(h,y) in enumerate(loader):
        if bi>=max_batches: break
        h=h.to(DEVICE)

        if name=="CIDM-Foundation-v1":
            out=model.model(h,horizon=HORIZON)
            d=out["diagnostics"]
            rows.append({
                "latent_energy_drift":float(d["latent_energy_drift"].detach()),
                "local_strength":float(d["local_strength"].detach()),
                "step_size":float(d["step_size"].detach()),
            })
        else:
            rows.append(model.architecture_diagnostics(h,HORIZON))

    keys=sorted({k for r in rows for k in r})
    return {k:float(np.mean([r[k] for r in rows if k in r])) for k in keys}



# 8. Frozen global learning rates

A2 不再进行新一轮调参。

直接复用 A1-R2 已经通过相同预算选择的 architecture-global LR：

```text
CIDM-Core          1e-4
CIDM-Complete      1e-3
PDEBench-FNO       3e-3
NeuralOperator-FNO 3e-3
Transolver         3e-4
```

这本身也是一次跨任务训练超参泛化测试。


In [ ]:

# ============================================================
# 8. Frozen architecture-global LR
# ============================================================
LR_SELECTED={
    "CIDM-Foundation-v1":1e-4,
    "CIDM-Complete":1e-3,
    "PDEBench-FNO":3e-3,
    "NeuralOperator-FNO":3e-3,
    "Transolver":3e-4,
}
assert set(LR_SELECTED)==set(MODELS)
save_json(LR_SELECTED,OUT/"A2_selected_lr.json")
pd.DataFrame([
    {"model":m,"lr":LR_SELECTED[m]}
    for m in MODELS
]).to_csv(OUT/"A2_selected_lr.csv",index=False)
REPORT["selected_lr"]=LR_SELECTED



# 9. A2 3-seed broad formal benchmark

正式：

\[
5\ models \times 8\ tasks \times 3\ seeds
=
120\ runs
\]

统一 convergence contract：

```text
MAX_UPDATES      = 2500
MIN_UPDATES      = 1000
EVAL_EVERY       = 100
PATIENCE_EVALS   = 6
REL_MIN_DELTA    = 0.1%
```

保留 deterministic step-wise batch plan。

### HF 持久化
- 本地每 100 step 写 latest checkpoint；
- 默认每 500 step 同步 HF Dataset；
- 每一组 run 结束后立即上传；
- 新 Colab 自动恢复 completed/unfinished run。

因此 A2 不依赖单次 Colab 最长运行时间。


In [ ]:

# ============================================================
# 9. A2 deterministic training + HF persistence
# ============================================================
def normalized_split_tensors(task,split):
    c=CACHES[task]
    norm=NORMALIZERS[task]
    h=c["splits"][split]["history"].float()
    y=c["splits"][split]["target"].float()

    mh=norm.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=norm.std.view(1,1,-1,*([1]*(h.ndim-3)))
    return (
        ((h-mh)/sh).contiguous(),
        ((y-mh)/sh).contiguous(),
    )

TRAIN_TENSORS={
    task:normalized_split_tensors(task,"train")
    for task in TASKS
}

def deterministic_batch_plan(n,batch,seed,steps):
    rng=np.random.default_rng(seed)
    plan=[]
    while len(plan)<steps:
        perm=rng.permutation(n)
        usable=(len(perm)//batch)*batch
        perm=perm[:usable]
        if usable==0:
            raise RuntimeError(f"n={n} smaller than batch={batch}")
        for j in range(0,usable,batch):
            plan.append(perm[j:j+batch].astype(np.int64))
            if len(plan)>=steps:
                break
    return plan

def train_a2_to_convergence(
    model,task,va,lr,seed,tag,latest_path
):
    model=model.to(DEVICE).float()

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=MAX_UPDATES,
        eta_min=lr*0.1
    )

    train_h,train_y=TRAIN_TENSORS[task]
    batch=BATCH_1D if task_spec(task)[0]==1 else BATCH_2D
    plan=deterministic_batch_plan(
        len(train_h),batch,seed,MAX_UPDATES
    )

    start_step=0
    raw_best=float("inf")
    raw_best_step=0
    best_sd=None
    convergence_ref=float("inf")
    bad_evals=0
    history=[]
    elapsed_before=0.0

    if latest_path.exists():
        ck=torch.load(
            latest_path,map_location="cpu",weights_only=False
        )
        load_portable_state_dict(model,ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])

        start_step=int(ck["step"])
        raw_best=float(ck["raw_best"])
        raw_best_step=int(ck["raw_best_step"])
        best_sd=ck.get("best_model")
        convergence_ref=float(
            ck.get("convergence_ref",raw_best)
        )
        bad_evals=int(ck.get("bad_evals",0))
        history=list(ck.get("history",[]))
        elapsed_before=float(
            ck.get("elapsed_train_seconds",0.0)
        )

        print(
            f"[A2 exact resume] {tag} "
            f"step={start_step}, best={raw_best:.6g}@{raw_best_step}"
        )

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0=time.perf_counter()
    stop_reason="MAX_UPDATES"

    for s in range(start_step+1,MAX_UPDATES+1):
        idx=torch.as_tensor(plan[s-1],dtype=torch.long)

        h=train_h.index_select(0,idx).to(
            DEVICE,non_blocking=True
        )
        y=train_y.index_select(0,idx).to(
            DEVICE,non_blocking=True
        )

        model.train()
        opt.zero_grad(set_to_none=True)

        p=model.rollout(h,HORIZON)
        loss=F.mse_loss(p,y)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"{tag}: non-finite loss at step {s}"
            )

        loss.backward()

        bad=[
            n for n,q in model.named_parameters()
            if q.grad is not None
            and not torch.isfinite(q.grad).all()
        ]
        if bad:
            raise FloatingPointError(
                f"{tag}: non-finite gradients {bad[:8]}"
            )

        grad_norm=float(
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),1.0
            )
        )
        opt.step()
        sched.step()

        should_stop=False

        if s==1 or s%EVAL_EVERY==0 or s==MAX_UPDATES:
            vr=evaluate_model(model,va,task)["rmse"]

            if vr<raw_best:
                raw_best=vr
                raw_best_step=s
                best_sd=portable_state_dict(model)

            if not math.isfinite(convergence_ref):
                convergence_ref=vr
                bad_evals=0
            else:
                threshold=convergence_ref*(1.0-REL_MIN_DELTA)
                if vr<threshold:
                    convergence_ref=vr
                    bad_evals=0
                elif s>=MIN_UPDATES:
                    bad_evals+=1

            history.append({
                "step":s,
                "train_normalized_mse":float(loss.detach()),
                "val_rmse_original":float(vr),
                "raw_best_val_rmse":float(raw_best),
                "raw_best_step":int(raw_best_step),
                "convergence_ref":float(convergence_ref),
                "bad_evals":int(bad_evals),
                "lr":float(opt.param_groups[0]["lr"]),
                "grad_norm_preclip":grad_norm,
            })

            print(
                f"[{tag}] {s}/{MAX_UPDATES} "
                f"val={vr:.6g} "
                f"best={raw_best:.6g}@{raw_best_step} "
                f"bad={bad_evals}/{PATIENCE_EVALS}"
            )

            if (
                s>=MIN_UPDATES
                and bad_evals>=PATIENCE_EVALS
            ):
                stop_reason="CONVERGED_PATIENCE"
                should_stop=True

        if (
            s%100==0
            or s==MAX_UPDATES
            or should_stop
        ):
            torch.cuda.synchronize()
            elapsed_total=(
                elapsed_before
                +(time.perf_counter()-t0)
            )

            torch.save({
                "step":s,
                "model":portable_state_dict(model),
                "best_model":best_sd,
                "raw_best":raw_best,
                "raw_best_step":raw_best_step,
                "convergence_ref":convergence_ref,
                "bad_evals":bad_evals,
                "optimizer":opt.state_dict(),
                "scheduler":sched.state_dict(),
                "history":history,
                "elapsed_train_seconds":elapsed_total,
                "run_id":tag,
                "task":task,
                "seed":seed,
            },latest_path)

            if (
                HF_ENABLED
                and (
                    s%HF_CHECKPOINT_EVERY==0
                    or s==MAX_UPDATES
                    or should_stop
                )
            ):
                hf_sync_a2_checkpoint(
                    latest_path,tag,s
                )

        if should_stop:
            break

    torch.cuda.synchronize()
    elapsed_total=(
        elapsed_before
        +(time.perf_counter()-t0)
    )
    peak=int(torch.cuda.max_memory_allocated())

    actual_updates=(
        history[-1]["step"] if history else start_step
    )

    if best_sd is None:
        raw_best=evaluate_model(model,va,task)["rmse"]
        raw_best_step=actual_updates
        best_sd=portable_state_dict(model)

    load_portable_state_dict(model,best_sd)

    return model,{
        "best_val_rmse":float(raw_best),
        "best_val_step":int(raw_best_step),
        "actual_updates":int(actual_updates),
        "stop_reason":stop_reason,
        "train_gpu_seconds":float(elapsed_total),
        "peak_vram_bytes":peak,
        "history":history,
    }

initial_state=scan_a2_progress()
print(
    f"[A2 START] completed="
    f"{initial_state['completed']}/{initial_state['total_expected']} "
    f"checkpoints={initial_state['latest_checkpoints']}"
)

FORMAL=[]

for seed in SEEDS:
    for task in TASKS:
        dim,c,S=task_spec(task)
        batch=BATCH_1D if dim==1 else BATCH_2D
        _,va,te=loaders_for(task,batch,seed)

        for name in MODELS:
            run_id=f"A2__{name}__{task}__s{seed}"
            result_path=OUT/f"{run_id}_result.json"
            history_path=OUT/f"{run_id}_history.csv"
            latest_path=CKPT/f"{run_id}_latest.pt"

            if result_path.exists():
                try:
                    row=json.loads(
                        result_path.read_text(
                            encoding="utf-8"
                        )
                    )
                except Exception:
                    row={}
                if row.get("status")=="COMPLETED":
                    FORMAL.append(row)
                    print("[A2 completed skip]",run_id)
                    continue

            seed_all(seed)
            cfg=MATCH[task][name]["config"]
            m=build_by_config(task,name,cfg)
            lr=LR_SELECTED[name]

            print("\n"+"="*108)
            print(
                run_id,
                "params",count_params(m),
                "lr",lr,
                "checkpoint",latest_path.exists()
            )

            write_a2_manifest(
                run_id,None,"RUN_START"
            )

            try:
                m,ti=train_a2_to_convergence(
                    m,task,va,lr,seed,
                    run_id,latest_path
                )

                if ti["history"]:
                    pd.DataFrame(
                        ti["history"]
                    ).to_csv(
                        history_path,index=False
                    )

                ev=evaluate_model(
                    m,te,task,
                    save_examples=(seed==SEEDS[0]),
                    example_tag=f"{task}__{name}"
                )
                lat=measure_latency(m,te)
                arch_diag=architecture_specific_diagnostics(
                    m,te,name
                )

                row={
                    "seed":seed,
                    "task":task,
                    "family":TASKS[task]["family"],
                    "model":name,
                    "status":"COMPLETED",
                    "params":count_params(m),
                    "lr":lr,
                    "best_val_rmse":ti["best_val_rmse"],
                    "best_val_step":ti["best_val_step"],
                    "actual_updates":ti["actual_updates"],
                    "stop_reason":ti["stop_reason"],
                    "train_gpu_seconds":ti["train_gpu_seconds"],
                    "peak_vram_bytes":ti["peak_vram_bytes"],
                    **ev,
                    **lat,
                    **arch_diag,
                }
                save_json(row,result_path)

            except Exception as e:
                row={
                    "seed":seed,
                    "task":task,
                    "family":TASKS[task]["family"],
                    "model":name,
                    "status":"FAILED",
                    "params":count_params(m),
                    "lr":lr,
                    "error":repr(e),
                }
                save_json(row,result_path)

                if (
                    latest_path.exists()
                    and not history_path.exists()
                ):
                    try:
                        ck=torch.load(
                            latest_path,
                            map_location="cpu",
                            weights_only=False
                        )
                        if ck.get("history"):
                            pd.DataFrame(
                                ck["history"]
                            ).to_csv(
                                history_path,index=False
                            )
                    except Exception:
                        pass

            FORMAL.append(row)
            print(row)

            # Per-run immediate HF persistence.
            hf_sync_a2_finished(
                run_id,
                result_path,
                history_path,
                latest_path,
            )

            if row.get("status")=="COMPLETED":
                latest_path.unlink(missing_ok=True)

            del m
            torch.cuda.empty_cache()
            gc.collect()

# Build cumulative result table from disk.
rows=[]
for p in sorted(OUT.glob("A2__*_result.json")):
    try:
        rows.append(
            json.loads(
                p.read_text(encoding="utf-8")
            )
        )
    except Exception:
        pass

formal_df=pd.DataFrame(rows)
formal_df.to_csv(
    OUT/"A2_run_results.csv",
    index=False
)
display(formal_df)

required=len(SEEDS)*len(TASKS)*len(MODELS)
completed=int(
    (formal_df.status=="COMPLETED").sum()
) if len(formal_df) else 0

print("A2 completion:",completed,"/",required)

write_a2_manifest(
    None,None,"TRAINING_LOOP_END"
)

if completed!=required:
    raise RuntimeError(
        f"A2 incomplete ({completed}/{required}). "
        "Restart Colab and run notebook from top: "
        "HF Dataset restores completed runs and latest checkpoint."
    )

REPORT["formal_runs"]=formal_df.to_dict("records")



# 10. A2 broad-dynamics analysis

3 seeds 不用于强行追求显著性。

本轮核心分析：
1. task × model mean ± std；
2. 每个 task 对最佳模型归一化；
3. 跨 8 task mean normalized score；
4. Core / Complete 对每类动力系统的 3-seed win count；
5. Complete 相对 Core 的改善率；
6. 每个 architecture 在不同 dynamics family 的行为画像。

A1-R2-R1 的 10-seed 结果继续承担论文主统计检验。


In [ ]:

# ============================================================
# 10. A2 statistics / regime analysis
# ============================================================
df=formal_df[formal_df.status=="COMPLETED"].copy()

summary=df.groupby(
    ["task","family","model"]
).agg(
    rmse_mean=("rmse","mean"),
    rmse_std=("rmse","std"),
    mae_mean=("mae","mean"),
    relative_l2_mean=("relative_l2","mean"),
    rollout_growth_mean=("rollout_growth","mean"),
    logspec_low_mean=("logspec_rmse_low","mean"),
    logspec_mid_mean=("logspec_rmse_mid","mean"),
    logspec_high_mean=("logspec_rmse_high","mean"),
    updates_mean=("actual_updates","mean"),
    train_seconds_mean=("train_gpu_seconds","mean"),
    latency_ms_mean=("latency_ms_median","mean"),
    peak_vram_mean=("peak_vram_bytes","mean"),
).reset_index()

summary.to_csv(
    OUT/"A2_summary_by_task_model.csv",
    index=False
)

# task-normalized RMSE ratio
mins=summary.groupby("task")["rmse_mean"].min().to_dict()
summary["task_normalized_rmse_ratio"]=summary.apply(
    lambda r:r.rmse_mean/mins[r.task],
    axis=1
)

overall=summary.groupby("model").agg(
    mean_task_normalized_rmse=(
        "task_normalized_rmse_ratio","mean"
    ),
    median_task_normalized_rmse=(
        "task_normalized_rmse_ratio","median"
    ),
    mean_rollout_growth=(
        "rollout_growth_mean","mean"
    ),
    mean_logspec_high=(
        "logspec_high_mean","mean"
    ),
    mean_latency_ms=(
        "latency_ms_mean","mean"
    ),
    mean_train_seconds=(
        "train_seconds_mean","mean"
    ),
    mean_peak_vram=(
        "peak_vram_mean","mean"
    ),
).reset_index().sort_values(
    "mean_task_normalized_rmse"
)

summary.to_csv(
    OUT/"A2_summary_with_normalized_score.csv",
    index=False
)
overall.to_csv(
    OUT/"A2_overall_task_normalized.csv",
    index=False
)

def paired_descriptive(reference,comparators):
    rows=[]
    for task in TASKS:
        ref=df[
            (df.task==task)
            &(df.model==reference)
        ][["seed","rmse"]].rename(
            columns={"rmse":"rmse_ref"}
        )
        for name in comparators:
            cmp=df[
                (df.task==task)
                &(df.model==name)
            ][["seed","rmse"]].rename(
                columns={"rmse":"rmse_cmp"}
            )
            m=ref.merge(cmp,on="seed")
            diff=(
                m.rmse_cmp-m.rmse_ref
            ).to_numpy()
            rows.append({
                "task":task,
                "family":TASKS[task]["family"],
                "reference":reference,
                "comparator":name,
                "mean_rmse_difference_cmp_minus_ref":
                    float(diff.mean()),
                "std_difference":
                    float(diff.std(ddof=1))
                    if len(diff)>1 else np.nan,
                "reference_lower_rmse_seed_count":
                    int((diff>0).sum()),
                "n_seeds":len(diff),
            })
    return pd.DataFrame(rows)

others=[
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "Transolver",
]

core_pairs=paired_descriptive(
    "CIDM-Foundation-v1",
    others+["CIDM-Complete"]
)
complete_pairs=paired_descriptive(
    "CIDM-Complete",
    others+["CIDM-Foundation-v1"]
)

core_pairs.to_csv(
    OUT/"A2_pairwise_vs_core.csv",
    index=False
)
complete_pairs.to_csv(
    OUT/"A2_pairwise_vs_complete.csv",
    index=False
)

# Core vs Complete direct table.
cc=[]
for task in TASKS:
    a=df[
        (df.task==task)
        &(df.model=="CIDM-Foundation-v1")
    ][["seed","rmse"]].rename(
        columns={"rmse":"core_rmse"}
    )
    b=df[
        (df.task==task)
        &(df.model=="CIDM-Complete")
    ][["seed","rmse"]].rename(
        columns={"rmse":"complete_rmse"}
    )
    m=a.merge(b,on="seed")
    for _,r in m.iterrows():
        cc.append({
            "task":task,
            "family":TASKS[task]["family"],
            "seed":int(r.seed),
            "core_rmse":float(r.core_rmse),
            "complete_rmse":float(r.complete_rmse),
            "complete_vs_core_pct":
                100*(r.complete_rmse-r.core_rmse)
                /r.core_rmse,
        })
cc_df=pd.DataFrame(cc)
cc_df.to_csv(
    OUT/"A2_Core_vs_Complete_seedwise.csv",
    index=False
)

# Dynamics regime map: normalized error + rank.
regime=summary[[
    "task","family","model",
    "rmse_mean","rmse_std",
    "task_normalized_rmse_ratio"
]].copy()
regime["rank_within_task"]=regime.groupby(
    "task"
)["rmse_mean"].rank(method="min")
regime.to_csv(
    OUT/"A2_dynamics_regime_map.csv",
    index=False
)

display(overall)
display(regime.sort_values(["task","rank_within_task"]))

REPORT["summary"]=summary.to_dict("records")
REPORT["overall"]=overall.to_dict("records")
REPORT["pairwise_vs_core"]=core_pairs.to_dict("records")
REPORT["pairwise_vs_complete"]=complete_pairs.to_dict("records")



# 11. A2 publication figures

延续前面确定的顶级期刊风格：

- 白底；
- serif / 数学字体；
- 细轴；
- individual seed 点可见；
- 不用 dashboard/card；
- 统一模型颜色；
- PDF/SVG/600dpi PNG；
- 同步 CSV。

A2 最重要的是一张 **Model × Dynamics Regime Heatmap**，
直接回答“哪类动力系统最适合 CIDM-Core / Complete”。


In [ ]:

# ============================================================
# 11. A2 publication-grade figures
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "figure.facecolor":"white",
    "axes.facecolor":"white",
    "savefig.facecolor":"white",
    "font.family":"serif",
    "font.serif":[
        "Times New Roman",
        "Nimbus Roman",
        "DejaVu Serif",
    ],
    "mathtext.fontset":"stix",
    "font.size":9,
    "axes.titlesize":9,
    "axes.labelsize":9,
    "xtick.labelsize":7.5,
    "ytick.labelsize":8,
    "legend.fontsize":7.5,
    "axes.linewidth":0.8,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "pdf.fonttype":42,
    "ps.fonttype":42,
})

MODEL_ORDER=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "Transolver",
]
MODEL_LABEL={
    "CIDM-Foundation-v1":"CIDM-Core",
    "CIDM-Complete":"CIDM-Complete",
    "PDEBench-FNO":"PDEBench FNO",
    "NeuralOperator-FNO":"NeuralOperator FNO",
    "Transolver":"Transolver",
}
MODEL_COLOR={
    "CIDM-Foundation-v1":"#1f4e79",
    "CIDM-Complete":"#b2182b",
    "PDEBench-FNO":"#4d4d4d",
    "NeuralOperator-FNO":"#2b8cbe",
    "Transolver":"#6a51a3",
}

TASK_LABEL={
    "advection_1d":"Advection",
    "burgers_1d":"Burgers",
    "diff_sorp_1d":"Diff.-Sorption",
    "reacdiff_1d":"1D Reac.-Diff.",
    "cfd_1d":"1D CFD",
    "reacdiff_2d":"2D Reac.-Diff.",
    "swe_2d":"Shallow Water",
    "cfd_2d":"2D CFD",
}

def savefig_all(fig,stem):
    for ext in ["pdf","svg","png"]:
        kw={"bbox_inches":"tight"}
        if ext=="png":
            kw["dpi"]=600
        fig.savefig(FIG/f"{stem}.{ext}",**kw)
    plt.close(fig)

# ------------------------------------------------------------
# Fig A2-01: Model × dynamics regime heatmap
# value = task-normalized RMSE (1.0 = task best)
# ------------------------------------------------------------
mat=regime.pivot(
    index="model",
    columns="task",
    values="task_normalized_rmse_ratio"
).loc[MODEL_ORDER,list(TASKS)]

fig,ax=plt.subplots(figsize=(8.2,2.7))
im=ax.imshow(
    mat.values,
    aspect="auto",
    cmap="YlOrRd",
    vmin=1.0,
    vmax=max(2.5,float(np.nanpercentile(mat.values,90))),
)
ax.set_xticks(np.arange(len(TASKS)))
ax.set_xticklabels(
    [TASK_LABEL[t] for t in TASKS],
    rotation=35,
    ha="right"
)
ax.set_yticks(np.arange(len(MODEL_ORDER)))
ax.set_yticklabels([MODEL_LABEL[m] for m in MODEL_ORDER])

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val=mat.iloc[i,j]
        ax.text(
            j,i,f"{val:.2f}",
            ha="center",va="center",
            fontsize=7,
            color="white" if val>1.8 else "black",
        )

cb=fig.colorbar(im,ax=ax,pad=0.015)
cb.set_label("Task-normalized RMSE")
ax.set_title("Broad dynamics regime map (lower is better)")
fig.tight_layout()

mat.reset_index().to_csv(
    FIGDATA/"Fig_A2_01_regime_heatmap.csv",
    index=False
)
savefig_all(fig,"Fig_A2_01_regime_heatmap")

# ------------------------------------------------------------
# Fig A2-02: 8 task panels, mean±std + individual seeds
# ------------------------------------------------------------
fig,axes=plt.subplots(2,4,figsize=(10.5,5.0))
axes=axes.ravel()
figrows=[]

for ax,task in zip(axes,TASKS):
    x=np.arange(len(MODEL_ORDER))
    q=summary[
        summary.task==task
    ].set_index("model").loc[MODEL_ORDER]

    means=q.rmse_mean.values
    stds=q.rmse_std.values

    ax.bar(
        x,means,
        color=[MODEL_COLOR[m] for m in MODEL_ORDER],
        edgecolor="black",
        linewidth=0.35,
        width=0.72,
        alpha=0.88,
    )
    ax.errorbar(
        x,means,yerr=stds,
        fmt="none",ecolor="black",
        capsize=2,elinewidth=0.7
    )

    for j,m in enumerate(MODEL_ORDER):
        vals=df[
            (df.task==task)
            &(df.model==m)
        ].rmse.values
        jitter=np.linspace(-0.08,0.08,len(vals))
        ax.scatter(
            np.full(len(vals),j)+jitter,
            vals,
            s=9,
            facecolor="white",
            edgecolor="black",
            linewidth=0.45,
            zorder=5,
        )
        for v in vals:
            figrows.append({
                "task":task,
                "model":m,
                "rmse":float(v),
            })

    ax.set_title(TASK_LABEL[task])
    ax.set_xticks(x)
    ax.set_xticklabels(
        [MODEL_LABEL[m] for m in MODEL_ORDER],
        rotation=55,ha="right"
    )
    ax.set_ylabel("RMSE")
    ax.set_yscale("log")

fig.tight_layout(w_pad=1.0,h_pad=1.0)
pd.DataFrame(figrows).to_csv(
    FIGDATA/"Fig_A2_02_task_rmse_panels.csv",
    index=False
)
savefig_all(fig,"Fig_A2_02_task_rmse_panels")

# ------------------------------------------------------------
# Fig A2-03: Core vs Complete paired task means
# ------------------------------------------------------------
fig,ax=plt.subplots(figsize=(6.6,3.0))
task_x=np.arange(len(TASKS))
core_means=[]
comp_means=[]

for task in TASKS:
    core_means.append(
        float(summary[
            (summary.task==task)
            &(summary.model=="CIDM-Foundation-v1")
        ].rmse_mean.iloc[0])
    )
    comp_means.append(
        float(summary[
            (summary.task==task)
            &(summary.model=="CIDM-Complete")
        ].rmse_mean.iloc[0])
    )

for i,(a,b) in enumerate(zip(core_means,comp_means)):
    ax.plot(
        [i-0.12,i+0.12],
        [a,b],
        color="#9e9e9e",
        lw=0.8,
        zorder=1,
    )

ax.scatter(
    task_x-0.12,core_means,
    color=MODEL_COLOR["CIDM-Foundation-v1"],
    s=28,label="CIDM-Core",zorder=3
)
ax.scatter(
    task_x+0.12,comp_means,
    color=MODEL_COLOR["CIDM-Complete"],
    s=28,label="CIDM-Complete",zorder=3
)
ax.set_yscale("log")
ax.set_xticks(task_x)
ax.set_xticklabels(
    [TASK_LABEL[t] for t in TASKS],
    rotation=35,ha="right"
)
ax.set_ylabel("RMSE")
ax.legend()
fig.tight_layout()

pd.DataFrame({
    "task":list(TASKS),
    "core_rmse":core_means,
    "complete_rmse":comp_means,
}).to_csv(
    FIGDATA/"Fig_A2_03_core_vs_complete.csv",
    index=False
)
savefig_all(fig,"Fig_A2_03_core_vs_complete")

# ------------------------------------------------------------
# Fig A2-04: Accuracy-latency Pareto
# ------------------------------------------------------------
fig,ax=plt.subplots(figsize=(3.9,3.0))
for _,r in overall.iterrows():
    m=r.model
    ax.scatter(
        r.mean_latency_ms,
        r.mean_task_normalized_rmse,
        s=46,
        color=MODEL_COLOR[m],
        edgecolor="black",
        linewidth=0.45,
    )
    ax.annotate(
        MODEL_LABEL[m],
        (r.mean_latency_ms,r.mean_task_normalized_rmse),
        xytext=(4,3),
        textcoords="offset points",
        fontsize=7,
    )

ax.set_xlabel("Median 6-step latency (ms)")
ax.set_ylabel("Mean task-normalized RMSE")
fig.tight_layout()
overall.to_csv(
    FIGDATA/"Fig_A2_04_accuracy_latency.csv",
    index=False
)
savefig_all(fig,"Fig_A2_04_accuracy_latency")

# ------------------------------------------------------------
# Fig A2-05: high-band log-spectrum error heatmap
# ------------------------------------------------------------
smat=summary.pivot(
    index="model",
    columns="task",
    values="logspec_high_mean"
).loc[MODEL_ORDER,list(TASKS)]

fig,ax=plt.subplots(figsize=(8.2,2.7))
im=ax.imshow(
    smat.values,
    aspect="auto",
    cmap="Blues",
)
ax.set_xticks(np.arange(len(TASKS)))
ax.set_xticklabels(
    [TASK_LABEL[t] for t in TASKS],
    rotation=35,ha="right"
)
ax.set_yticks(np.arange(len(MODEL_ORDER)))
ax.set_yticklabels(
    [MODEL_LABEL[m] for m in MODEL_ORDER]
)
cb=fig.colorbar(im,ax=ax,pad=0.015)
cb.set_label("High-band log-spectrum RMSE")
ax.set_title("Spectral fidelity across dynamics regimes")
fig.tight_layout()

smat.reset_index().to_csv(
    FIGDATA/"Fig_A2_05_spectral_heatmap.csv",
    index=False
)
savefig_all(fig,"Fig_A2_05_spectral_heatmap")

# ------------------------------------------------------------
# Fig A2-06: representative examples
# Advection, 1D CFD, SWE, 2D CFD
# ------------------------------------------------------------
REP_TASKS=[
    "advection_1d",
    "cfd_1d",
    "swe_2d",
    "cfd_2d",
]
rep_rows=[]

fig,axes=plt.subplots(4,4,figsize=(8.4,7.8))

for i,task in enumerate(REP_TASKS):
    # Pick channel 0 for a stable common visualization.
    exact=torch.load(
        PRED/f"{task}__CIDM-Complete.pt",
        map_location="cpu",
        weights_only=False
    )["target"]
    pc=torch.load(
        PRED/f"{task}__CIDM-Complete.pt",
        map_location="cpu",
        weights_only=False
    )["pred"]
    pcore=torch.load(
        PRED/f"{task}__CIDM-Foundation-v1.pt",
        map_location="cpu",
        weights_only=False
    )["pred"]

    # best external by task mean RMSE
    ext=summary[
        (summary.task==task)
        &(~summary.model.str.startswith("CIDM"))
    ].sort_values("rmse_mean").iloc[0].model

    pext=torch.load(
        PRED/f"{task}__{ext}.pt",
        map_location="cpu",
        weights_only=False
    )["pred"]

    if exact.ndim==3:  # [K,C,X]
        li=HORIZON-1
        x=np.arange(exact.shape[-1])
        axes[i,0].plot(
            x,exact[li,0],
            color="black",lw=1.2
        )
        axes[i,0].set_title("Exact" if i==0 else "")
        axes[i,1].plot(
            x,pcore[li,0],
            color=MODEL_COLOR["CIDM-Foundation-v1"],lw=1.1
        )
        axes[i,1].set_title("CIDM-Core" if i==0 else "")
        axes[i,2].plot(
            x,pc[li,0],
            color=MODEL_COLOR["CIDM-Complete"],lw=1.1
        )
        axes[i,2].set_title("CIDM-Complete" if i==0 else "")
        axes[i,3].plot(
            x,pext[li,0],
            color=MODEL_COLOR[ext],lw=1.1
        )
        axes[i,3].set_title(
            MODEL_LABEL[ext] if i==0 else ""
        )
        for j in range(4):
            axes[i,j].set_xlabel("x")
    else:  # [K,C,H,W]
        li=HORIZON-1
        arrs=[
            exact[li,0].numpy(),
            pcore[li,0].numpy(),
            pc[li,0].numpy(),
            pext[li,0].numpy(),
        ]
        vmax=max(float(np.max(np.abs(a))) for a in arrs)
        titles=[
            "Exact","CIDM-Core","CIDM-Complete",
            MODEL_LABEL[ext]
        ]
        for j,a in enumerate(arrs):
            axes[i,j].imshow(
                a,cmap="RdBu_r",
                vmin=-vmax,vmax=vmax,
                origin="lower"
            )
            axes[i,j].set_xticks([])
            axes[i,j].set_yticks([])
            if i==0:
                axes[i,j].set_title(titles[j])

    axes[i,0].set_ylabel(TASK_LABEL[task])

fig.tight_layout()
savefig_all(fig,"Fig_A2_06_representative_predictions")

print("A2 figures:",FIG)



# 12. A2 final gate

通过条件：

- frozen CIDM Core / Complete hash PASS；
- official external commits PASS；
- 8 compact public PDEBench task caches PASS；
- 40/40 preflight；
- 40 parameter matches within 500K ±5%；
- frozen global LR；
- 120/120 formal runs；
- figures and regime-map files generated；
- HF progress manifest produced。

不要求任何 CIDM 版本获胜。


In [ ]:

# ============================================================
# 12. A2 final audit / gate
# ============================================================
required_runs=len(SEEDS)*len(TASKS)*len(MODELS)

required_figs=[
    "Fig_A2_01_regime_heatmap",
    "Fig_A2_02_task_rmse_panels",
    "Fig_A2_03_core_vs_complete",
    "Fig_A2_04_accuracy_latency",
    "Fig_A2_05_spectral_heatmap",
    "Fig_A2_06_representative_predictions",
]
fig_ok=all(
    all(
        (FIG/f"{stem}.{ext}").exists()
        for ext in ["pdf","svg","png"]
    )
    for stem in required_figs
)

checks={
    "frozen_cidm_core_exact":
        REPORT["cidm_freeze"]["exact"],
    "frozen_cidm_complete_exact":
        REPORT["cidm_complete_freeze"]["exact"],
    "official_commits_exact":
        all(x["exact"] for x in checkout),
    "eight_task_caches":
        len(CACHES)==8,
    "parameter_match_40of40":
        len(match_df)==len(TASKS)*len(MODELS)
        and bool(match_df["ok"].all()),
    "preflight_40of40":
        len(pre_df)==len(TASKS)*len(MODELS)
        and bool(pre_df["pass"].all()),
    "three_seed_protocol":
        len(SEEDS)==3,
    "formal_120of120":
        len(df)==required_runs,
    "regime_map_written":
        (OUT/"A2_dynamics_regime_map.csv").exists(),
    "figures_complete":
        fig_ok,
    "hf_manifest_written":
        (OUT/"A2_progress_manifest.json").exists(),
}

verdict=(
    "PASS_A2_BROAD_DYNAMICS_BENCHMARK"
    if all(checks.values())
    else "A2_INCOMPLETE_OR_UNFAIR"
)

REPORT["gate"]=checks
REPORT["verdict"]=verdict
REPORT["finished_at"]=time.strftime(
    "%Y-%m-%d %H:%M:%S"
)

save_json(
    REPORT,
    OUT/"CIDM_Foundation_A2_REPORT.json"
)

lines=[
    "CIDM Foundation A2 — Broad Dynamics Benchmark",
    "="*86,
    f"VERDICT: {verdict}",
    "",
]
for k,v in checks.items():
    lines.append(
        f"[{'PASS' if v else 'FAIL'}] {k}"
    )
lines += [
    "",
    "This gate certifies broad-dynamics benchmark completeness/fairness only.",
    "It does not require CIDM-Core or CIDM-Complete to win.",
    "",
    "Next if PASS:",
    "Proceed to A3 CIDM-Complete Component Attribution using representative regimes.",
]

gate_path=OUT/"A2_gate_report.txt"
gate_path.write_text(
    "\n".join(lines),
    encoding="utf-8"
)
print(gate_path.read_text(encoding="utf-8"))



# 13. Package + HF final synchronization

最终包包含：
- official provenance；
- data manifest；
- compact-cache protocol；
- parameter matching；
- all 120 run JSON / history；
- regime analysis；
- figures / figure-data；
- prediction examples；
- gate report。

不把 raw PDEBench 大文件塞进结果 ZIP；
compact cache 已独立保存在 HF Dataset 的 A2 prefix 中。


In [ ]:

# ============================================================
# 13. A2 package + final HF snapshot + auto-download
# ============================================================
zip_path=OUT/"CIDM_Foundation_A2_BroadDynamics_RESULTS.zip"

with zipfile.ZipFile(
    zip_path,"w",zipfile.ZIP_DEFLATED
) as zf:
    for p in OUT.rglob("*"):
        if not p.is_file() or p==zip_path:
            continue
        if (
            "sources" in p.parts
            or "checkpoints" in p.parts
        ):
            continue
        zf.write(
            p,
            p.relative_to(OUT)
        )

sha=sha256_file(zip_path)
(
    OUT/"CIDM_Foundation_A2_BroadDynamics_RESULTS.zip.sha256"
).write_text(
    sha+"\n",
    encoding="utf-8"
)

print("ZIP:",zip_path)
print("SHA256:",sha)

if HF_ENABLED:
    try:
        write_a2_manifest(
            None,None,"FINAL_PACKAGE"
        )
        HF_API.upload_folder(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            folder_path=str(OUT),
            path_in_repo=HF_PROGRESS_PREFIX,
            ignore_patterns=[
                "sources/*",
                "checkpoints/*",
            ],
            commit_message="[A2] final broad-dynamics benchmark package",
        )
        print(
            f"[HF A2 FINAL PASS] "
            f"https://huggingface.co/datasets/"
            f"{HF_DATASET_REPO_ID}/tree/main/{HF_PROGRESS_PREFIX}"
        )
    except Exception as e:
        print("[HF A2 FINAL WARNING]",repr(e))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print(
        "Auto-download unavailable outside Colab:",
        repr(e)
    )
    print("Result:",zip_path.resolve())
